# Bat Swing Plane - 3D Trajectory Pipeline

Reconstructs a cricket bat's 3D swing trajectory from a single 2D video: crease detection and stabilization, bat segmentation (SAM 3.1), depth estimation, keypoint extraction, and a 3D swept-plane visualization.

**Requirements**
- GPU required: minimum 32GB VRAM
- Jupyter with ipykernel installed (checked in Section 6)

Run the sections in order, top to bottom.


## 1. Install dependencies

Create the environment from `requirements_gpu.yml`, then register it as a Jupyter kernel and select it for this notebook:

```
conda env create -f requirements_gpu.yml -n bat_env
conda run -n bat_env python -m ipykernel install --user --name bat_env --display-name "bat_env"
```

Select the `bat_env` kernel above before running the cells below. Note: `requirements_gpu.yml` pins a PyTorch nightly build for very new GPU architectures; if that exact build is no longer on the nightly index, install the latest matching nightly for your CUDA version instead.


## 2. Imports


In [1]:
import os, sys, logging, time, gc, warnings, subprocess, tempfile, shutil
warnings.filterwarnings("ignore")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["LOG_LEVEL"] = "WARNING"
logging.basicConfig(level=logging.WARNING, force=True)

import cv2
import numpy as np
import torch
torch.backends.cudnn.enabled = False

from PIL import Image
import transformers
transformers.logging.set_verbosity_error()

from transformers import pipeline as hf_pipeline, AutoProcessor, AutoModelForZeroShotObjectDetection
from ultralytics import YOLO

# NOTE: These assume sam2 and sam3 are installed in your environment or present in the working directory
try:
    from sam3.model_builder import build_sam3_multiplex_video_predictor
    from sam2.build_sam import build_sam2_video_predictor
except ImportError as e:
    print("WARNING: Could not import SAM2/SAM3. Please ensure they are installed/available:", e)


## 3. Configs


In [2]:

# ══════════════════════════════════════════════════════════════════
# USER CONFIGURATION — Edit these paths before running
# ══════════════════════════════════════════════════════════════════

# Input video path (can be a Google Drive path or local path)
INPUT_VIDEO = "input/vk_flick_1.mp4"

# Output directory for all results
OUTPUT_DIR  = "outputs"

# Visualization mode: True = save all debug videos, False = only final swing_analysis.mp4
VISUALIZATION = True

# Z-scale factor to align monocular depth with X/Y pixel coordinates
Z_SCALE_FACTOR_OVERRIDE = 60000

# ══════════════════════════════════════════════════════════════════
# BELOW ARE THE REPOSITORY CONSTANTS (formerly config.py)
# ══════════════════════════════════════════════════════════════════



# ═══════════════════════════════════════════════════════════════
# 1. MODEL PATHS & DEVICES
# ═══════════════════════════════════════════════════════════════
YOLO_OBJ_WEIGHTS       = "weights/yolo26x.pt"
YOLO_POSE_WEIGHTS      = "weights/yolo26x-pose.pt"
SAM2_CONFIG            = "sam2_hiera_l.yaml"
SAM2_WEIGHTS           = "weights/sam2_hiera_large.pt"
SAM3_WEIGHTS           = "weights/sam3.1_multiplex.pt"
DEPTH_MODEL_NAME       = "depth-anything/Depth-Anything-V2-Large-hf"
DINO_MODEL_NAME        = "IDEA-Research/grounding-dino-tiny"

# ═══════════════════════════════════════════════════════════════
# 2. GLOBAL OUTPUT
# ═══════════════════════════════════════════════════════════════
OUTPUT_WIDTH           = 1280
OUTPUT_HEIGHT          = 1280
DEFAULT_FPS            = 30.0
STITCH_FPS             = 16.0
DEPTH_BATCH_SIZE       = 8

# ═══════════════════════════════════════════════════════════════
# 3. PITCH STABILIZER  (src/pitch_stabilizer.py)
# ═══════════════════════════════════════════════════════════════
PITCH_TARGET_LEFT_X    = 250
PITCH_TARGET_RIGHT_X   = 1030
PITCH_TARGET_LINE_Y    = 800
PITCH_QUERY            = "the vertical white line . white boundary line"
PITCH_BOX_THRESHOLD    = 0.30
PITCH_TEXT_THRESHOLD    = 0.30
PITCH_SAM2_NUM_POINTS  = 5        # points per crease line for SAM2 prompt
PITCH_BOTTOM_MASK_FRAC = 0.84     # black out below this fraction of frame height
PITCH_EDGE_MARGIN      = 80       # black out left/right edge pixels
# DINO detection filter heuristics (pitch_stabilizer.py lines 47-54)
PITCH_TOP_FRAC         = 0.60     # line must be in top 60% of frame
PITCH_MAX_WIDTH_FRAC   = 0.15     # max box width as fraction of frame width
PITCH_MIN_HEIGHT_FRAC  = 0.03     # min box height as fraction of frame height
PITCH_MAX_HEIGHT_FRAC  = 0.15     # max box height as fraction of frame height
PITCH_MIN_ASPECT       = 1.5      # box height must be > width * this
PITCH_MIN_HORZ_GAP     = 0.10     # minimum horizontal gap between two lines (frac of frame width)

# ═══════════════════════════════════════════════════════════════
# 4. VIDEO TRIMMER  (src/video_trimmer.py)
# ═══════════════════════════════════════════════════════════════
BAT_CLASS_ID           = 34       # COCO class id for 'baseball bat'
BAT_DETECT_CONF        = 0.15     # minimum bat detection confidence
PERSON_DETECT_CONF     = 0.30     # minimum person detection confidence
TRIM_RESIZE_W          = 640
TRIM_RESIZE_H          = 360
TRIM_BLOCK_GAP         = 5        # max gap between frames in a contiguous block
TRIM_BATCH_SIZE        = 8
BAT_PROXIMITY_MARGIN   = 60       # pixels margin for bat-person intersection in trimmer

# ═══════════════════════════════════════════════════════════════
# 5. TRACKING  (src/video_trimmer.py)
# ═══════════════════════════════════════════════════════════════
W_IOU                  = 0.6
W_BOTTOM_DIST          = 0.2
W_ASPECT_RATIO         = 0.15
W_REL_POSITION         = 0.05
TRACK_MAX_GAP          = 30
IOU_MIN_THRESH         = 0.05
ASPECT_RATIO_TOLERANCE = 0.25
AREA_RATIO_HARD_GATE   = 0.33     # reject match if area ratio < this

# ═══════════════════════════════════════════════════════════════
# 7. SAM SEGMENTATION  (src/sam_segmentation.py)
# ═══════════════════════════════════════════════════════════════
SAM_BAT_PROMPT             = "bat"
SAM_WICKET_PROMPT          = "wicket"
SAM_MIN_MASK_AREA          = 300
SAM_GDINO_THRESHOLD        = 0.20
SAM_GDINO_RAW_THRESHOLD    = 0.10    # low threshold for raw detection logging
SAM_GDINO_SCAN_THRESHOLD   = 0.01    # ultra-low threshold for full-frame scan
SAM_WICKET_ZONE_PAD        = 1.3     # padding factor for wicket exclusion zone
SAM_WICKET_OVERLAP_THRESH  = 0.70    # blob overlap to classify as wicket
SAM_REF_AREA_MIN_KEEP      = 0.40    # fraction of reference area to keep blob
SAM_DEBUG_FPS              = 30.0

# ═══════════════════════════════════════════════════════════════
# 8. KEYPOINT EXTRACTOR  (src/keypoint_extractor.py)
# ═══════════════════════════════════════════════════════════════
KP_YOLO_CONF               = 0.35
KP_WRIST_CONF              = 0.30
KP_GRIP_FALLBACK_FRAMES    = 30
KP_MIN_MASK_PIXELS         = 20     # PCA needs at least this many pixels
KP_GRIP_DISTANCE_TIEBREAK  = 10     # px — grip distance must differ by this to be decisive
KP_END_FRAC                = 0.20   # fraction of axis for end-width voting
KP_TIP_FRAC                = 0.15   # fraction of axis for tip centroid
KP_HANDLE_FRAC             = 0.15   # fraction of axis for handle centroid

# ═══════════════════════════════════════════════════════════════
# 9. DEPTH EXTRACTOR  (src/depth_extractor.py)
# ═══════════════════════════════════════════════════════════════
DEPTH_EPSILON              = 1e-6   # inverse-depth epsilon to avoid div-by-zero

# ═══════════════════════════════════════════════════════════════
# 10. 3D TRAJECTORY  (src/trajectory_3d.py)
# ═══════════════════════════════════════════════════════════════
TRAJ_SMOOTH_WINDOW_X       = 5
TRAJ_SMOOTH_WINDOW_Y       = 5
TRAJ_SMOOTH_WINDOW_Z       = 7
TRAJ_SPLINE_UPSAMPLE       = 10      # 1=disabled, >1=Catmull-Rom smoothing multiplier
TRAJ_DEPTH_PATCH_RADIUS    = 3
import os
_z_scale = os.environ.get("TRAJ_Z_SCALE_FACTOR")
# ── Z Scale Factor — CALIBRATE THIS AGAINST YOUR GROUND TRUTH ─────────────────
# Monocular depth models output relative/inverse-depth values in the range ~[0, 1].
# TRAJ_Z_SCALE_FACTOR multiplies those raw depth values to bring them into a
# physically meaningful scale (e.g., pixels or centimetres) that is visually
# comparable to the X/Y pixel dimensions of the frame.
#
TRAJ_Z_SCALE_FACTOR        = int(_z_scale) if _z_scale else 100_000
TRAJ_MIN_VALID_OBS         = 4       # minimum valid 3D points for RTS
TRAJ_OUTLIER_SIGMA         = 3.0     # depth outlier rejection sigma
TRAJ_CMAP_TIP              = "inferno"
TRAJ_CMAP_HANDLE           = "viridis"
TRAJ_RIBBON_ALPHA          = 0.5
TRAJ_RIBBON_COLOR_LOW      = "#7B00D4"   # violet
TRAJ_RIBBON_COLOR_HIGH     = "#FFD700"   # yellow
# Each entry is (elevation, azimuth, label).
# The label is used as the output filename: traj_3d_{label}.mp4
# Changing the angle values does NOT change the filename — only changing the label does.
TRAJ_VIEW_ANGLES           = [(10, -80, "front"), (10, -10, "side")]
TRAJ_RENDER_DPI            = 200
TRAJ_RENDER_FIG_SIZE       = (10, 7)
TRAJ_BG_COLOR              = "#0d0d0d"
TRAJ_AXIS_BG_COLOR         = "#111111"
TRAJ_AXIS_TICK_MAJOR       = 100     # MultipleLocator step
TRAJ_FFMPEG_FRAMERATE      = 25
TRAJ_BOX_ZOOM              = 1.0       # Scale the 3D box inside the plot window
TRAJ_MARGIN_LEFT           = 0.0       # Left margin (0 = absolute edge)
TRAJ_MARGIN_RIGHT          = 0.85      # Right margin (leave 15% for colorbar)
TRAJ_MARGIN_BOTTOM         = 0.15      # Bottom margin (0.05 gives labels breathing room)
TRAJ_MARGIN_TOP            = 1.0       # Top margin (1.0 = absolute top edge)

# Kalman filter parameters (9-state constant-acceleration model)
KALMAN_Q_DIAG              = [5., 5., 8., 10., 10., 12., 1., 1., 2.]
KALMAN_R_DIAG              = [15., 15., 25.]
KALMAN_INIT_COV_SCALE      = 50.0

# ═══════════════════════════════════════════════════════════════
# 11. DEBUG VIDEO RENDERING  (pipeline.py)
# ═══════════════════════════════════════════════════════════════
DEBUG_TIP_COLOR            = (0, 255, 255)       # cyan  (BGR)
DEBUG_HANDLE_COLOR         = (0, 140, 255)       # orange (BGR)
DEBUG_MASK_COLOR_RGB       = (0, 255, 0)         # green (RGB float)
DEBUG_DOT_RADIUS           = 8
DEBUG_CLAHE_CLIP_LIMIT     = 5.0
DEBUG_CLAHE_TILE_SIZE      = (8, 8)
DEBUG_SAT_MULTIPLIER       = 1.8
DEBUG_UNSHARP_STRENGTH     = 2.0     # addWeighted alpha
DEBUG_UNSHARP_SIGMA        = 3.0     # GaussianBlur sigma
DEBUG_BLACK_THRESH         = 10      # pixel < this → considered black border

# ── Depth preview color range (windowing) ─────────────────────────────────────
# Controls how the depth colormap (TURBO) range is chosen in the debug depth videos.
# This is DISPLAY ONLY — it does not affect the depth data used for 3D trajectory.
#   "bat"   → focus the full color range on the bat's narrow depth band (high contrast on the bat)
#   "scene" → legacy behaviour: mean ± 3σ across ALL non-padding pixels
DEPTH_VIZ_MODE             = "bat"
DEPTH_VIZ_PCT_LO           = 2.0     # lower percentile of bat-pixel depths for window floor
DEPTH_VIZ_PCT_HI           = 98.0    # upper percentile of bat-pixel depths for window ceiling
DEPTH_VIZ_PAD              = 0.10     # expand the window by this fraction of its span on each side
DEPTH_VIZ_MIN_BAT_PIXELS   = 50      # need at least this many bat pixels total, else fall back to "scene"



In [3]:
if Z_SCALE_FACTOR_OVERRIDE is not None:
    os.environ["TRAJ_Z_SCALE_FACTOR"] = str(Z_SCALE_FACTOR_OVERRIDE)

_BASE = os.path.abspath(".")
input_video = INPUT_VIDEO if os.path.isabs(INPUT_VIDEO) else os.path.join(_BASE, INPUT_VIDEO)
out_dir = OUTPUT_DIR if os.path.isabs(OUTPUT_DIR) else os.path.join(_BASE, OUTPUT_DIR)
os.makedirs(out_dir, exist_ok=True)


In [4]:
_TIP_COLOR    = DEBUG_TIP_COLOR
_HANDLE_COLOR = DEBUG_HANDLE_COLOR
_MASK_COLOR   = np.array(DEBUG_MASK_COLOR_RGB, dtype=np.float32)
_DOT_R        = DEBUG_DOT_RADIUS
_FONT         = cv2.FONT_HERSHEY_SIMPLEX


def _draw_frame_label(img_bgr, fidx, n_total):
    """Draw a clearly visible 'Frame: X / N' label in the top-left corner."""
    label = f"Frame: {fidx} / {n_total - 1}"
    scale, thick = 0.8, 2
    (tw, th), baseline = cv2.getTextSize(label, _FONT, scale, thick)
    pad = 8
    # dark filled rectangle behind the text
    cv2.rectangle(img_bgr, (0, 0), (tw + pad * 2, th + baseline + pad * 2), (0, 0, 0), -1)
    cv2.putText(img_bgr, label, (pad, th + pad),
                _FONT, scale, (255, 255, 255), thick, cv2.LINE_AA)

def _draw_bat_points(img_bgr, tip_xy, handle_xy, fidx, depth_map=None, show_patch=False):
    """Draw tip (cyan) and handle (orange) dots/boxes on a BGR frame without text labels next to them."""
    for row, color, label in [
        (next((r for r in tip_xy    if int(r[0]) == fidx), None), _TIP_COLOR,    "tip"),
        (next((r for r in handle_xy if int(r[0]) == fidx), None), _HANDLE_COLOR, "handle"),
    ]:
        if row is None:
            continue
        x, y = row[1], row[2]
        if np.isnan(x) or np.isnan(y):
            continue
        cx, cy = int(round(x)), int(round(y))

        # Draw dot
        cv2.circle(img_bgr, (cx, cy), _DOT_R, color, -1)
        cv2.circle(img_bgr, (cx, cy), _DOT_R + 2, (0, 0, 0), 2)

        # If show_patch is True, draw a 7x7 box around the point showing the sampled patch
        if show_patch:
            cv2.rectangle(img_bgr, (cx - 3, cy - 3), (cx + 3, cy + 3), color, 1)

def _draw_corner_labels(img_bgr, tip_xy, handle_xy, fidx, depth_map=None):
    """Draw tip and handle status/names in the upper right corner with different colors."""
    H, W = img_bgr.shape[:2]
    scale = 0.5
    thick = 2

    # Position in the top-right corner.
    y_pos = 30
    x_pos_right = W - 20

    for row, color, label in [
        (next((r for r in tip_xy    if int(r[0]) == fidx), None), _TIP_COLOR,    "Tip"),
        (next((r for r in handle_xy if int(r[0]) == fidx), None), _HANDLE_COLOR, "Handle"),
    ]:
        status_str = f"{label}: "
        if row is None:
            status_str += "MISSING"
            text_color = (0, 0, 255) # Red
        else:
            x, y = row[1], row[2]
            if np.isnan(x) or np.isnan(y):
                status_str += "MISSING (NaN)"
                text_color = (0, 0, 255) # Red
            else:
                if depth_map is not None:
                    z = sample_depth(depth_map, x, y, TRAJ_DEPTH_PATCH_RADIUS)
                    if z != 0:
                        status_str += f"VALID (z={z:.4f})"
                        text_color = (0, 255, 0) # Green
                    else:
                        status_str += f"INVALID (z={z:.4f})"
                        text_color = (0, 0, 255) # Red
                else:
                    status_str += "DETECTED"
                    text_color = color

        # Draw background rectangle for text readability
        (tw, th), _ = cv2.getTextSize(status_str, _FONT, scale, thick)
        cv2.rectangle(img_bgr, (x_pos_right - tw - 10, y_pos - th - 5), (x_pos_right + 5, y_pos + 5), (0, 0, 0), -1)
        cv2.putText(img_bgr, status_str, (x_pos_right - tw - 5, y_pos), _FONT, scale, text_color, thick, cv2.LINE_AA)
        y_pos += th + 15

def _draw_depth_legend(img_bgr, d_min, d_max):
    """Draw a vertical colorbar legend for the depth map on the right side of the image."""
    H, W = img_bgr.shape[:2]

    # Legend dimensions
    leg_w = max(20, W // 40)
    leg_h = int(H * 0.4)

    # Create gradient 0-255 (top is 255/red, bottom is 0/blue)
    gradient = np.linspace(255, 0, leg_h, dtype=np.uint8)
    gradient = np.tile(gradient[:, None], (1, leg_w))

    # Apply colormap
    gradient_bgr = cv2.applyColorMap(gradient, cv2.COLORMAP_TURBO)

    # Placement: bottom right corner, above any tracking text if possible
    pad = 20
    x_offset = W - leg_w - pad
    y_offset = H - leg_h - pad - 40 # Leave some space at bottom

    # Overlay gradient
    img_bgr[y_offset:y_offset+leg_h, x_offset:x_offset+leg_w] = gradient_bgr

    # Draw border
    cv2.rectangle(img_bgr, (x_offset, y_offset), (x_offset+leg_w, y_offset+leg_h), (255, 255, 255), 2)

    # Text labels
    scale = 0.6
    thick = 2
    label_max = f"Max: {d_max:.1f}"
    label_min = f"Min: {d_min:.1f}"

    # Top text (Max)
    (tw_max, th_max), _ = cv2.getTextSize(label_max, _FONT, scale, thick)
    cv2.putText(img_bgr, label_max, (x_offset - tw_max - 10, y_offset + th_max),
                _FONT, scale, (0, 0, 0), thick + 1, cv2.LINE_AA)
    cv2.putText(img_bgr, label_max, (x_offset - tw_max - 10, y_offset + th_max),
                _FONT, scale, (255, 255, 255), thick, cv2.LINE_AA)

    # Bottom text (Min)
    (tw_min, th_min), _ = cv2.getTextSize(label_min, _FONT, scale, thick)
    cv2.putText(img_bgr, label_min, (x_offset - tw_min - 10, y_offset + leg_h),
                _FONT, scale, (0, 0, 0), thick + 1, cv2.LINE_AA)
    cv2.putText(img_bgr, label_min, (x_offset - tw_min - 10, y_offset + leg_h),
                _FONT, scale, (255, 255, 255), thick, cv2.LINE_AA)

def _draw_depth_histogram(img_bgr, depth_map, global_d_min, global_d_max, n_bins=50):
    """Draw a depth-value histogram in the bottom-left corner of img_bgr.
    Bars are coloured with TURBO. X-axis is clipped to mean ± 3σ to remove
    outliers; the colorbar legend still shows the true global range."""
    H, W = img_bgr.shape[:2]

    valid = depth_map[(depth_map != 0) & np.isfinite(depth_map)].ravel().astype(np.float64)
    if valid.size == 0:
        return

    # Clip to mean ± 3σ so outlier pixels don't collapse the histogram
    mu  = float(valid.mean())
    sig = float(valid.std())
    lo  = mu - 3.0 * sig
    hi  = mu + 3.0 * sig
    clipped = valid[(valid >= lo) & (valid <= hi)]
    if clipped.size == 0:
        clipped, lo, hi = valid, float(valid.min()), float(valid.max())

    hist, _ = np.histogram(clipped, bins=n_bins, range=(lo, hi))
    max_count = int(hist.max())
    if max_count == 0:
        return

    plot_w = int(W * 0.35)
    plot_h = int(H * 0.18)
    pad    = 15
    left   = pad
    top    = H - plot_h - pad - 18   # 18 px for x-axis labels below

    # Semi-transparent black background
    bg = img_bgr.copy()
    cv2.rectangle(bg, (left - 6, top - 22), (left + plot_w + 6, top + plot_h + 20), (0, 0, 0), -1)
    cv2.addWeighted(bg, 0.55, img_bgr, 0.45, 0, dst=img_bgr)

    # Bars coloured with TURBO mapped to the clipped range position
    bin_w = max(1, plot_w // n_bins)
    for i, count in enumerate(hist):
        if count == 0:
            continue
        bar_h  = max(1, int((count / max_count) * plot_h))
        bx     = left + i * bin_w
        by_bot = top + plot_h
        by_top = by_bot - bar_h
        # Map bin position within clipped range → global range → TURBO color
        bin_val   = lo + (i / n_bins) * (hi - lo)
        color_frac = (bin_val - global_d_min) / max(global_d_max - global_d_min, 1e-6)
        color_pos = int(np.clip(color_frac, 0, 1) * 255) if np.isfinite(color_frac) else 128
        color     = cv2.applyColorMap(np.array([[color_pos]], dtype=np.uint8),
                                      cv2.COLORMAP_TURBO)[0, 0].tolist()
        cv2.rectangle(img_bgr, (bx, by_top), (bx + bin_w - 1, by_bot), color, -1)

    # Plot border
    cv2.rectangle(img_bgr, (left, top), (left + plot_w, top + plot_h), (160, 160, 160), 1)

    fs, ft = 0.38, 1
    mid_val = (lo + hi) / 2.0

    # X-axis tick labels (clipped range)
    cv2.putText(img_bgr, f"{lo:.2f}",
                (left, top + plot_h + 13), _FONT, fs, (210, 210, 210), ft, cv2.LINE_AA)
    cv2.putText(img_bgr, f"{mid_val:.2f}",
                (left + plot_w // 2 - 18, top + plot_h + 13), _FONT, fs, (210, 210, 210), ft, cv2.LINE_AA)
    cv2.putText(img_bgr, f"{hi:.2f}",
                (left + plot_w - 32, top + plot_h + 13), _FONT, fs, (210, 210, 210), ft, cv2.LINE_AA)

    # Y-axis: max count at top, 0 at bottom
    cv2.putText(img_bgr, str(max_count),
                (left + 2, top + 10), _FONT, fs, (210, 210, 210), ft, cv2.LINE_AA)
    cv2.putText(img_bgr, "0",
                (left + 2, top + plot_h - 2), _FONT, fs, (210, 210, 210), ft, cv2.LINE_AA)

    # Title
    cv2.putText(img_bgr, "Depth distribution",
                (left, top - 6), _FONT, fs, (210, 210, 210), ft, cv2.LINE_AA)

def _make_side_by_side(video_a, video_b, out_path,
                       label_a="Stabilized", label_b="3D Trajectory"):
    """Write a side-by-side MP4: video_a on the left, video_b on the right.
    Both are scaled to the same height. If frame counts differ the shorter
    video holds its last frame until the longer one finishes."""
    cap_a = cv2.VideoCapture(video_a)
    cap_b = cv2.VideoCapture(video_b)
    if not cap_a.isOpened() or not cap_b.isOpened():
        print("  WARNING: [merge] could not open input videos")
        cap_a.release(); cap_b.release()
        return

    # from config import OUTPUT_WIDTH, OUTPUT_HEIGHT

    # Enforce exactly 30 FPS for all outputs to ensure concat compatibility
    fps_out = 30.0

    n_a     = int(cap_a.get(cv2.CAP_PROP_FRAME_COUNT))
    n_b     = int(cap_b.get(cv2.CAP_PROP_FRAME_COUNT))
    n_out   = max(n_a, n_b)

    def _read_first(cap):
        ret, f = cap.read()
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        return f if ret else None

    first_a = _read_first(cap_a)
    first_b = _read_first(cap_b)
    if first_a is None or first_b is None:
        cap_a.release(); cap_b.release()
        return

    def _pad_to_exact(frame, target_w, target_h):
        fh, fw = frame.shape[:2]
        scale = min(target_w / fw, target_h / fh)
        nw, nh = int(fw * scale), int(fh * scale)
        resized = cv2.resize(frame, (nw, nh), interpolation=cv2.INTER_LINEAR)
        top = (target_h - nh) // 2
        bottom = target_h - nh - top
        left = (target_w - nw) // 2
        right = target_w - nw - left
        return cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=[0,0,0])

    W_out = OUTPUT_WIDTH * 2
    H = OUTPUT_HEIGHT

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_path, fourcc, fps_out, (W_out, H))

    def _label(frame, text):
        out = frame.copy()
        (tw, _th), _ = cv2.getTextSize(text, _FONT, 2.5, 8)
        cx = max(10, (out.shape[1] - tw) // 2)
        cv2.putText(out, text, (cx, 80), _FONT, 2.5, (0, 0, 0),   8, cv2.LINE_AA)
        cv2.putText(out, text, (cx, 80), _FONT, 2.5, (255, 255, 255), 4, cv2.LINE_AA)
        return out

    last_a = _pad_to_exact(first_a, OUTPUT_WIDTH, H)
    last_b = _pad_to_exact(first_b, OUTPUT_WIDTH, H)

    for _ in range(n_out):
        ret_a, f_a = cap_a.read()
        ret_b, f_b = cap_b.read()
        if ret_a:
            last_a = _pad_to_exact(f_a, OUTPUT_WIDTH, H)
        if ret_b:
            last_b = _pad_to_exact(f_b, OUTPUT_WIDTH, H)
        writer.write(np.hstack([_label(last_a, label_a), _label(last_b, label_b)]))

    writer.release()
    cap_a.release()
    cap_b.release()
    print(f"  [merge] side-by-side saved: {os.path.basename(out_path)}")

def _make_3in1_video(rgb_path, view1_path, view2_path, out_path,
                     label_rgb="Stabilized", label_v1="3D Front View", label_v2="3D Side View"):
    """Write a 3-in-1 MP4 video horizontally: RGB | Front View | Side View"""
    cap_rgb = cv2.VideoCapture(rgb_path)
    cap_v1 = cv2.VideoCapture(view1_path)
    cap_v2 = cv2.VideoCapture(view2_path)

    if not (cap_rgb.isOpened() and cap_v1.isOpened() and cap_v2.isOpened()):
        print("  WARNING: [merge] could not open input videos")
        cap_rgb.release(); cap_v1.release(); cap_v2.release()
        return

    # # from config import OUTPUT_WIDTH, OUTPUT_HEIGHT, STITCH_FPS
    fps_out = STITCH_FPS

    n_rgb = int(cap_rgb.get(cv2.CAP_PROP_FRAME_COUNT))
    n_v1 = int(cap_v1.get(cv2.CAP_PROP_FRAME_COUNT))
    n_out = max(n_rgb, n_v1)

    def _read_first(cap):
        ret, f = cap.read()
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        return f if ret else None

    first_rgb = _read_first(cap_rgb)
    first_v1 = _read_first(cap_v1)
    first_v2 = _read_first(cap_v2)

    if any(f is None for f in (first_rgb, first_v1, first_v2)):
        return

    def _pad_to_exact(frame, target_w, target_h):
        fh, fw = frame.shape[:2]
        scale = min(target_w / fw, target_h / fh)
        nw, nh = int(fw * scale), int(fh * scale)
        resized = cv2.resize(frame, (nw, nh), interpolation=cv2.INTER_LINEAR)
        top = (target_h - nh) // 2
        bottom = target_h - nh - top
        left = (target_w - nw) // 2
        right = target_w - nw - left
        return cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=[0,0,0])

    W_out = OUTPUT_WIDTH * 3
    H_out = OUTPUT_HEIGHT

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_path, fourcc, fps_out, (W_out, H_out))

    def _label(frame, text):
        out = frame.copy()
        (tw, _th), _ = cv2.getTextSize(text, _FONT, 2.5, 8)
        cx = max(10, (out.shape[1] - tw) // 2)
        cv2.putText(out, text, (cx, 80), _FONT, 2.5, (0, 0, 0),   8, cv2.LINE_AA)
        cv2.putText(out, text, (cx, 80), _FONT, 2.5, (255, 255, 255), 4, cv2.LINE_AA)
        return out

    last_rgb = _pad_to_exact(first_rgb, OUTPUT_WIDTH, OUTPUT_HEIGHT)
    last_v1 = _pad_to_exact(first_v1, OUTPUT_WIDTH, OUTPUT_HEIGHT)
    last_v2 = _pad_to_exact(first_v2, OUTPUT_WIDTH, OUTPUT_HEIGHT)

    for _ in range(n_out):
        ret_rgb, f_rgb = cap_rgb.read()
        ret_v1, f_v1 = cap_v1.read()
        ret_v2, f_v2 = cap_v2.read()

        if ret_rgb: last_rgb = _pad_to_exact(f_rgb, OUTPUT_WIDTH, OUTPUT_HEIGHT)
        if ret_v1: last_v1 = _pad_to_exact(f_v1, OUTPUT_WIDTH, OUTPUT_HEIGHT)
        if ret_v2: last_v2 = _pad_to_exact(f_v2, OUTPUT_WIDTH, OUTPUT_HEIGHT)

        row = np.hstack([_label(last_rgb, label_rgb), _label(last_v1, label_v1), _label(last_v2, label_v2)])
        writer.write(row)

    writer.release()
    cap_rgb.release(); cap_v1.release(); cap_v2.release()
    print(f"  [merge] swing analysis saved: {os.path.basename(out_path)}")

def _save_debug_videos(tracked_masks, rgb_frames, norm_depth,
                       tip_xy, handle_xy, out_dir, fps=30.0):
    """
    Write three debug MP4s into out_dir:

      bat_mask_keypoints.mp4 — anchored RGB + green bat mask + cyan tip + orange handle
      depth_colormap.mp4    — normalized depth (TURBO colormap) + same tip / handle points (with validity)
      depth_bat_mask.mp4    — normalized depth (TURBO colormap) + green bat mask overlay + points
    """
    n_frames = len(rgb_frames)
    H, W     = rgb_frames[0].shape[:2]
    depth_ok = norm_depth is not None and len(norm_depth) > 0
    n_depth  = len(norm_depth) if depth_ok else 0
    if depth_ok and n_depth != n_frames:
        print(f"  [depth] note: depth frames ({n_depth}) != rgb frames ({n_frames}), using min")
    if depth_ok:
        def _scene_window():
            # Legacy: mean ± 3σ across all valid (non-padding) pixels to clip outliers.
            # Cast to float64 before stats to avoid float32 overflow on large arrays.
            all_valid = np.concatenate([d[d != 0].ravel() for d in norm_depth]).astype(np.float64)
            if all_valid.size == 0:
                return 0.0, 1.0
            _mu  = float(np.mean(all_valid))
            _sig = float(np.std(all_valid))
            if np.isfinite(_mu) and np.isfinite(_sig) and _sig > 0:
                return _mu - 3.0 * _sig, _mu + 3.0 * _sig
            return float(np.percentile(all_valid, 1)), float(np.percentile(all_valid, 99))

        def _bat_window():
            # Focus the color range on the bat's narrow depth band.
            # Collect depth values that fall under the bat mask across all frames.
            bat_vals = []
            for fi in range(min(n_depth, n_frames)):
                m = tracked_masks.get(fi)
                if m is None or not m.any():
                    continue
                d = norm_depth[fi]
                # masks and depth maps can differ in resolution — align defensively
                if m.shape != d.shape:
                    m = cv2.resize(m.astype(np.uint8), (d.shape[1], d.shape[0]),
                                   interpolation=cv2.INTER_NEAREST).astype(bool)
                vals = d[m & (d != 0)]
                if vals.size:
                    bat_vals.append(vals.ravel())
            if not bat_vals:
                return None
            bat_all = np.concatenate(bat_vals).astype(np.float64)
            if bat_all.size < DEPTH_VIZ_MIN_BAT_PIXELS:
                return None
            lo = float(np.percentile(bat_all, DEPTH_VIZ_PCT_LO))
            hi = float(np.percentile(bat_all, DEPTH_VIZ_PCT_HI))
            if not (np.isfinite(lo) and np.isfinite(hi)) or hi <= lo:
                return None
            pad = (hi - lo) * DEPTH_VIZ_PAD
            return lo - pad, hi + pad

        window = _bat_window() if DEPTH_VIZ_MODE == "bat" else None
        if window is None:
            if DEPTH_VIZ_MODE == "bat":
                print("  [debug] depth viz: no usable bat pixels — falling back to scene range")
            window = _scene_window()
        global_d_min, global_d_max = window
        print(f"  [debug] depth viz range [{DEPTH_VIZ_MODE}]: "
              f"{global_d_min:.6g} → {global_d_max:.6g}")
        global_span = global_d_max - global_d_min if global_d_max > global_d_min else 1.0

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    rgb_writer   = cv2.VideoWriter(os.path.join(out_dir, "bat_mask_keypoints.mp4"),   fourcc, fps, (W, H))
    depth_writer = cv2.VideoWriter(os.path.join(out_dir, "depth_colormap.mp4"), fourcc, fps, (W, H)) if depth_ok else None
    depth_mask_writer = cv2.VideoWriter(os.path.join(out_dir, "depth_bat_mask.mp4"), fourcc, fps, (W, H)) if depth_ok else None

    print(f"  [debug] writing {n_frames} frames to {out_dir}")

    for fidx in range(n_frames):
        # ── RGB frame: mask overlay + points ─────────────────────────────────
        frame_f = rgb_frames[fidx].copy().astype(np.float32)
        bat_m   = tracked_masks.get(fidx)
        if bat_m is not None and bat_m.any():
            frame_f[bat_m] = frame_f[bat_m] * 0.5 + _MASK_COLOR * 0.5
        frame_bgr = cv2.cvtColor(frame_f.clip(0, 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
        _draw_bat_points(frame_bgr, tip_xy, handle_xy, fidx, depth_map=None, show_patch=False)
        _draw_corner_labels(frame_bgr, tip_xy, handle_xy, fidx, depth_map=None)
        _draw_frame_label(frame_bgr, fidx, n_frames)
        rgb_writer.write(frame_bgr)

        # ── Depth frames ─────────────────────────────────────────────────────
        if depth_ok and fidx < n_depth:
            d = norm_depth[fidx]
            d_u8  = np.clip(((d - global_d_min) / global_span * 255), 0, 255).astype(np.uint8)
            d_bgr = cv2.applyColorMap(d_u8, cv2.COLORMAP_TURBO)
            if d_bgr.shape[:2] != (H, W):
                d_bgr = cv2.resize(d_bgr, (W, H), interpolation=cv2.INTER_LINEAR)

            # Video 1: pure depth (TURBO) + legend + histogram
            d_only = d_bgr.copy()
            # _draw_depth_legend(d_only, global_d_min, global_d_max)
            # _draw_depth_histogram(d_only, d, global_d_min, global_d_max)
            _draw_frame_label(d_only, fidx, n_frames)
            depth_writer.write(d_only)

            # Video 2: pure depth (TURBO) + bat mask contour + tip/handle points + labels
            d_bgr_mask = d_bgr.copy()
            if bat_m is not None and bat_m.any():
                contours, _ = cv2.findContours(bat_m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(d_bgr_mask, contours, -1, (0, 255, 0), 2)
            _draw_bat_points(d_bgr_mask, tip_xy, handle_xy, fidx, depth_map=d, show_patch=True)
            _draw_corner_labels(d_bgr_mask, tip_xy, handle_xy, fidx, depth_map=d)
            _draw_frame_label(d_bgr_mask, fidx, n_frames)
            # _draw_depth_legend(d_bgr_mask, global_d_min, global_d_max)
            depth_mask_writer.write(d_bgr_mask)

    rgb_writer.release()
    print(f"  [debug] ✅ RGB        → {out_dir}/bat_mask_keypoints.mp4")

    if depth_writer is not None:
        depth_writer.release()
        depth_mask_writer.release()
        print(f"  [debug] ✅ Depth      → {out_dir}/depth_colormap.mp4")
        print(f"  [debug] ✅ Depth+Mask → {out_dir}/depth_bat_mask.mp4")
    else:
        print("  [debug] ⚠  Depth skipped (frame-count mismatch or norm_depth is None).")


## 4. Model weights check


In [5]:
import shutil
import urllib.request
from pathlib import Path

def _check_weight(path, label, url=None, hint=None):
    p = Path(path)
    if p.exists():
        print(f"  {label}: found")
        return
    if url:
        print(f"  {label}: missing, downloading...")
        p.parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(url, p)
            print(f"  {label}: downloaded")
        except Exception as e:
            print(f"  {label}: download failed ({e})")
    else:
        print(f"  {label}: missing.")
        if hint:
            print(f"    {hint}")

def _check_yolo(path, filename):
    p = Path(path)
    if p.exists():
        print(f"  {filename}: found")
        return
    print(f"  {filename}: missing, downloading via ultralytics...")
    try:
        from ultralytics import YOLO
        model = YOLO(filename)
        cached = Path(getattr(model, "ckpt_path", filename))
        p.parent.mkdir(parents=True, exist_ok=True)
        if cached.exists() and cached.resolve() != p.resolve():
            shutil.copy(cached, p)
        del model
        print(f"  {filename}: ready")
    except Exception as e:
        print(f"  {filename}: download failed ({e})")

print("Model weights check:")
_check_weight(
    SAM2_WEIGHTS, "sam2_hiera_large.pt",
    url="https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt",
)
_check_weight(
    SAM3_WEIGHTS, "sam3.1_multiplex.pt",
    hint="Download from https://huggingface.co/facebook/sam3.1 (~3.3 GB) and place it at the path above.",
)
_check_yolo(YOLO_OBJ_WEIGHTS, "yolo26x.pt")
_check_yolo(YOLO_POSE_WEIGHTS, "yolo26x-pose.pt")


Model weights check:
  sam2_hiera_large.pt: found
  sam3.1_multiplex.pt: found
  yolo26x.pt: found
  yolo26x-pose.pt: found


## 5. Load the input video


In [6]:
from IPython.display import Video, display

preview_path = os.path.join(out_dir, "input_preview.mp4")
print(f"Input video: {INPUT_VIDEO}")
if os.path.exists(input_video):
    subprocess.run(f'ffmpeg -y -i "{input_video}" -vcodec libx264 -f mp4 "{preview_path}" -loglevel quiet', shell=True)
    if os.path.exists(preview_path):
        display(Video(preview_path, embed=True, width=480))
else:
    print("File not found.")


Input video: input/vk_flick_1.mp4


## 6. Verify environment


In [7]:
import importlib.util

def _importable(pkg):
    return importlib.util.find_spec(pkg) is not None

CHECK_PACKAGES = [
    "torch", "torchvision", "cv2", "transformers", "ultralytics",
    "numpy", "scipy", "pykalman", "decord", "PIL", "matplotlib", "tqdm",
    "ipykernel", "sam2", "sam3",
]

print("Package check:")
missing = [pkg for pkg in CHECK_PACKAGES if not _importable(pkg)]
for pkg in CHECK_PACKAGES:
    print(f"  {pkg}: {'OK' if pkg not in missing else 'MISSING'}")

device_id = 0 if torch.cuda.is_available() else -1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nGPU check:")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"  device: {gpu_name}")
    print(f"  total memory: {total_gb:.1f} GB")
    if total_gb < 32:
        print("  WARNING: less than the required 32GB minimum.")
else:
    print("  WARNING: no CUDA GPU detected. This pipeline requires a GPU.")

if missing:
    print(f"\n{len(missing)} package(s) missing: {missing}")
else:
    print("\nAll checks passed.")


Package check:
  torch: OK
  torchvision: OK
  cv2: OK
  transformers: OK
  ultralytics: OK
  numpy: OK
  scipy: OK
  pykalman: OK
  decord: OK
  PIL: OK
  matplotlib: OK
  tqdm: OK
  ipykernel: OK
  sam2: OK
  sam3: OK

GPU check:
  device: NVIDIA GeForce RTX 5090
  total memory: 31.4 GB

All checks passed.


## 7. Crease line detection

This stage has three parts: load the raw video, auto-trim it down to the actual shot (cutting lead-up/follow-through), then use Grounding DINO + SAM 2 to find and track the two crease lines.


### 7a. Load raw frames


In [8]:
import cv2
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor


def load_frames(input_video):
    """Read all frames of a video into RAM (BGR). Returns (frames, fps)."""
    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        raise FileNotFoundError(f"Video not found: {input_video}")
    fps = cap.get(cv2.CAP_PROP_FPS) or DEFAULT_FPS

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(frame)
    cap.release()
    return frames, fps



### 7b. Auto-trim helper functions

YOLO detects the batsman and bat in every frame; a lightweight identity tracker follows the batsman across frames and scores each track by how often the bat is near their hands, so the trimmer can isolate the one track that's actually playing the shot.


In [9]:
import cv2
import numpy as np



def find_shot_boundaries(box_history):
    frames = sorted(box_history.keys())
    if not frames: return None, None

    blocks, current_block = [], [frames[0]]
    for f in frames[1:]:
        if f <= current_block[-1] + TRIM_BLOCK_GAP:
            current_block.append(f)
        else:
            blocks.append(current_block)
            current_block = [f]
    blocks.append(current_block)

    best_block = max(blocks, key=len)
    return best_block[0], best_block[-1]

def get_bat_boxes(results):
    bat_boxes = []
    for box in results.boxes:
        if int(box.cls[0]) == BAT_CLASS_ID and float(box.conf[0]) > BAT_DETECT_CONF:
            bat_boxes.append(tuple(map(int, box.xyxy[0])))
    return bat_boxes

def interpolate_boxes(boxes):
    """Fill None gaps in box list using linear interpolation (NO extrapolation)."""
    n = len(boxes)
    first_valid = next((i for i, b in enumerate(boxes) if b is not None), None)
    last_valid  = next((i for i, b in reversed(list(enumerate(boxes))) if b is not None), None)
    if first_valid is None:
        return boxes

    # Only interpolate internal gaps; leave edges as None for proper trimming
    for i in range(first_valid + 1, last_valid):
        if boxes[i] is None:
            prev_v = i - 1
            next_v = next(j for j in range(i + 1, last_valid + 1) if boxes[j] is not None)
            start_b, end_b = np.array(boxes[prev_v]), np.array(boxes[next_v])
            ratio = (i - prev_v) / (next_v - prev_v)
            boxes[i] = (start_b + (end_b - start_b) * ratio).tolist()
    return boxes

def check_intersection(box1, box2, margin=0):
    b1_x1, b1_y1, b1_x2, b1_y2 = box1
    b2_x1, b2_y1, b2_x2, b2_y2 = box2
    b1_x1 -= margin; b1_y1 -= margin
    b1_x2 += margin; b1_y2 += margin
    return not (b1_x2 < b2_x1 or b1_x1 > b2_x2 or b1_y2 < b2_y1 or b1_y1 > b2_y2)

def compute_iou(boxA, boxB):
    ax1,ay1,ax2,ay2 = boxA; bx1,by1,bx2,by2 = boxB
    ix1,iy1 = max(ax1,bx1), max(ay1,by1)
    ix2,iy2 = min(ax2,bx2), min(ay2,by2)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    if inter == 0: return 0.0
    return inter / ((ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter)

def compute_bottom_dist_score(boxA, boxB):
    height = max(boxA[3]-boxA[1], boxB[3]-boxB[1], 1.0)
    dist = (((boxA[0]+boxA[2])/2 - (boxB[0]+boxB[2])/2)**2 +
            (boxA[3] - boxB[3])**2) ** 0.5
    return max(0.0, 1.0 - dist/height)

def compute_aspect_score(boxA, boxB):
    def aspect(b): return max(b[3]-b[1],1.0)/max(b[2]-b[0],1.0)
    diff = abs(aspect(boxA) - aspect(boxB))
    if diff > ASPECT_RATIO_TOLERANCE:
        return max(0.0, 1.0 - (diff - ASPECT_RATIO_TOLERANCE) * 2.0)
    return 1.0

def compute_rel_position_score(track, det_box, all_track_boxes):
    if not all_track_boxes or track.get('last_box') is None: return 1.0
    t_cx = (track['last_box'][0]+track['last_box'][2])/2
    d_cx = (det_box[0]+det_box[2])/2
    violations, comparisons = 0, 0
    for _, ob in all_track_boxes.items():
        o_cx = (ob[0]+ob[2])/2
        if (t_cx < o_cx) != (d_cx < o_cx): violations += 1
        comparisons += 1
    return 1.0 if comparisons == 0 else 1.0 - (violations/comparisons)*0.5

def match_score(track, detection, all_track_last_boxes):
    tbox, dbox = track['last_box'], detection['box']
    iou = compute_iou(tbox, dbox)

    if iou < IOU_MIN_THRESH:
        return -1.0

    tarea = max(1, (tbox[2] - tbox[0]) * (tbox[3] - tbox[1]))
    darea = max(1, (dbox[2] - dbox[0]) * (dbox[3] - dbox[1]))
    area_ratio = min(tarea, darea) / max(tarea, darea)
    if area_ratio < AREA_RATIO_HARD_GATE:
        return -1.0

    return (W_IOU * iou +
            W_BOTTOM_DIST  * compute_bottom_dist_score(tbox, dbox) +
            W_ASPECT_RATIO * compute_aspect_score(tbox, dbox) +
            W_REL_POSITION * compute_rel_position_score(track, dbox, all_track_last_boxes))



### 7c. Auto-trim entry point

Runs YOLO-pose + YOLO-object detection over the whole clip in batches, builds identity tracks for every detected person, scores each track by bat proximity and hand closeness, and keeps only the contiguous block of frames belonging to the best-scoring track.


In [10]:
def trim_video(frames, yolo_pose_model, yolo_obj_model, batch_size=TRIM_BATCH_SIZE):
    """
    The new (test_stabilize.py) Stage 2 trimming logic.
    Uses multi-frame identity tracking and the area drop hard gate.
    """
    h, w = frames[0].shape[:2]
    small = [cv2.resize(f, (TRIM_RESIZE_W, TRIM_RESIZE_H)) for f in frames]
    sx, sy = w / TRIM_RESIZE_W, h / TRIM_RESIZE_H

    # --- STEP A: Extract (Batched) ---
    raw_frame_data = []
    for i in range(0, len(small), batch_size):
        batch_pose = yolo_pose_model(small[i:i+batch_size], verbose=False, conf=PERSON_DETECT_CONF)
        batch_obj  = yolo_obj_model(small[i:i+batch_size], verbose=False, conf=BAT_DETECT_CONF)
        for j in range(len(batch_pose)):
            res_pose = batch_pose[j]
            res_obj  = batch_obj[j]

            bat_boxes = get_bat_boxes(res_obj)

            persons = []
            if res_pose.keypoints is not None and len(res_pose.keypoints.xy) > 0:
                for k, box in enumerate(res_pose.boxes):
                    if int(box.cls[0]) != 0 or float(box.conf[0]) < PERSON_DETECT_CONF: continue
                    x1,y1,x2,y2 = map(int, box.xyxy[0])
                    kps = res_pose.keypoints.xy[k].cpu().numpy()
                    wrist_dist, kps_arr = None, kps
                    if len(kps) > 10:
                        lw, rw = kps[9], kps[10]
                        if lw[0] > 0 and rw[0] > 0:
                            wrist_dist = float(np.sqrt((lw[0]-rw[0])**2 + (lw[1]-rw[1])**2))
                    persons.append({'box': [x1,y1,x2,y2], 'wrist_dist': wrist_dist, 'kps': kps_arr})

            raw_frame_data.append({'persons': persons, 'bats': bat_boxes})

    # --- STEP B: Build identity tracks ---
    tracks = []
    track_id_counter = 1
    for fi, data in enumerate(raw_frame_data):
        unmatched = list(data['persons'])
        active = [t for t in tracks if fi - t['last_seen'] <= TRACK_MAX_GAP]
        all_last_boxes = {id(t): t['last_box'] for t in active if t.get('last_box') is not None}

        score_matrix = []
        for ti, track in enumerate(active):
            for di, det in enumerate(unmatched):
                s = match_score(track, det, all_last_boxes)
                if s >= 0:
                    score_matrix.append((s, ti, di))
        score_matrix.sort(reverse=True)

        matched_t, matched_d = set(), set()
        for s, ti, di in score_matrix:
            if ti in matched_t or di in matched_d: continue
            t = active[ti]; det = unmatched[di]
            t['frames'][fi] = det
            t['last_seen']  = fi
            t['last_box']   = det['box']
            matched_t.add(ti); matched_d.add(di)

        for di, det in enumerate(unmatched):
            if di not in matched_d:
                tracks.append({'id': track_id_counter, 'frames': {fi: det}, 'last_seen': fi, 'last_box': det['box']})
                track_id_counter += 1

    # --- STEP C: Score tracks ---
    def score_track(t):
        if not t['frames']: return 0
        bat_cnt, hands_score, avg_x = 0, 0.0, 0.0
        for fi, p in t['frames'].items():
            bb = p['box']
            avg_x += (bb[0]+bb[2])/2
            if any(check_intersection(bb, bat, margin=BAT_PROXIMITY_MARGIN) for bat in raw_frame_data[fi]['bats']):
                bat_cnt += 1
            if p.get('wrist_dist') is not None:
                hands_score += 1000.0 / (p['wrist_dist'] + 1.0)
        n = len(t['frames'])
        avg_x /= n
        return (bat_cnt * 100) + hands_score + n + (-abs(avg_x - TRIM_RESIZE_W/2))

    if not tracks:
        return None, None, None

    best_track = max(tracks, key=score_track)

    # --- STEP D: Interpolate and trim ---
    all_boxes = [best_track['frames'].get(i, None) for i in range(len(frames))]
    bb_only   = [p['box'] if p is not None else None for p in all_boxes]
    interp    = interpolate_boxes(bb_only)

    box_history = {}
    for fi, bb in enumerate(interp):
        if bb is None: continue
        x1,y1,x2,y2 = map(int, bb)
        box_history[fi] = (int(x1*sx), int(y1*sy), int(x2*sx), int(y2*sy))

    if not box_history:
        return None, None, None

    start_frame, end_frame = find_shot_boundaries(box_history)
    return start_frame, end_frame, box_history



### 7d. Detect crease lines & stabilize

Grounding DINO scans frames for two vertical crease-line boxes on a "golden" anchor frame, SAM 2 tracks their masks forward/backward through the whole clip, and every frame gets warped so the two crease points always land on the same fixed pixels.


In [11]:
import os
import cv2
import tempfile
import shutil
import numpy as np
import torch
from PIL import Image



TARGET_NORM_W = OUTPUT_WIDTH
TARGET_NORM_H = OUTPUT_HEIGHT
TARGET_LEFT_X = PITCH_TARGET_LEFT_X
TARGET_RIGHT_X = PITCH_TARGET_RIGHT_X
TARGET_PITCH_W = TARGET_RIGHT_X - TARGET_LEFT_X
TARGET_LINE_Y = PITCH_TARGET_LINE_Y

def extract_line_from_mask(mask_2d):
    ys, xs = np.where(mask_2d > 0)
    if len(xs) == 0: return None
    return (int(xs.mean()), int(ys.min()))

def stabilize(trimmed_frames, dino_processor, dino_model, sam2_predictor, device):
    """
    Takes a list of trimmed RGB frames and stabilizes the pitch using DINO and SAM2.
    Returns: (stab_frames, debug_frames)
    """
    h, w = trimmed_frames[0].shape[:2]
    golden_idx, golden_left, golden_right = None, None, None

    with torch.inference_mode():
        for fidx, frame in enumerate(trimmed_frames):
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            inp = dino_processor(images=pil, text=PITCH_QUERY, return_tensors="pt").to(device)
            out = dino_model(**inp)
            res = dino_processor.post_process_grounded_object_detection(
                out, inp.input_ids, threshold=PITCH_BOX_THRESHOLD,
                text_threshold=PITCH_TEXT_THRESHOLD, target_sizes=[(h, w)]
            )[0]

            valid = []
            for b in res["boxes"].cpu().numpy():
                bx1, by1, bx2, by2 = map(int, b)
                bw, bh = bx2-bx1, by2-by1
                in_top_60 = by2 < (h * PITCH_TOP_FRAC)
                if bw < w*PITCH_MAX_WIDTH_FRAC and bh > h*PITCH_MIN_HEIGHT_FRAC and bh < h*PITCH_MAX_HEIGHT_FRAC and bh > bw*PITCH_MIN_ASPECT and in_top_60:
                    valid.append({'box': b, 'area': bw*bh, 'cx': (bx1+bx2)/2})

            valid = sorted(valid, key=lambda x: x['area'])
            if len(valid) >= 2:
                b1 = valid[0]
                b2 = next((c for c in valid[1:] if abs(c['cx'] - b1['cx']) > w*PITCH_MIN_HORZ_GAP), None)
                if b2:
                    bl, br = (b1['box'], b2['box']) if b1['cx'] < b2['cx'] else (b2['box'], b1['box'])
                    golden_left  = {'x': (bl[0]+bl[2])/2, 'y_top': bl[1], 'y_bot': bl[3]}
                    golden_right = {'x': (br[0]+br[2])/2, 'y_top': br[1], 'y_bot': br[3]}
                    golden_idx = fidx
                    break

    if golden_idx is None:
        print("FAIL: DINO could not find pitch crease. Returning original frames.")
        orig = [cv2.resize(f, (TARGET_NORM_W, TARGET_NORM_H)) for f in trimmed_frames]
        return orig, trimmed_frames

    ram = "/dev/shm" if os.path.exists("/dev/shm") else None
    tmp_dir = tempfile.mkdtemp(dir=ram)
    for i, f in enumerate(trimmed_frames):
        cv2.imwrite(os.path.join(tmp_dir, f"{i:05d}.jpg"), f)

    segs = {}
    autocast_ctx = torch.autocast(device.type, dtype=torch.bfloat16) if device.type == "cuda" else None

    try:
        if autocast_ctx: autocast_ctx.__enter__()

        state = sam2_predictor.init_state(video_path=tmp_dir)

        for obj_id, side in [(1, golden_left), (2, golden_right)]:
            ys = np.linspace(side['y_top'], side['y_bot'], PITCH_SAM2_NUM_POINTS)
            pts = np.array([[side['x'], y] for y in ys], dtype=np.float32)
            sam2_predictor.add_new_points_or_box(inference_state=state, frame_idx=golden_idx,
                                       obj_id=obj_id, points=pts, labels=np.ones(PITCH_SAM2_NUM_POINTS, dtype=np.int32))

        for fi, oids, masks in sam2_predictor.propagate_in_video(state, start_frame_idx=golden_idx, reverse=False):
            segs[fi] = {oid: (masks[i] > 0.0).cpu().numpy().squeeze() for i, oid in enumerate(oids)}
        if golden_idx > 0:
            for fi, oids, masks in sam2_predictor.propagate_in_video(state, start_frame_idx=golden_idx, reverse=True):
                segs[fi] = {oid: (masks[i] > 0.0).cpu().numpy().squeeze() for i, oid in enumerate(oids)}

    finally:
        if 'state' in locals(): sam2_predictor.reset_state(state)
        if autocast_ctx: autocast_ctx.__exit__(None, None, None)
        shutil.rmtree(tmp_dir, ignore_errors=True)

    stab_frames = []
    debug_frames = []
    lx, lyt = golden_left['x'],  golden_left['y_top']
    rx, ryt = golden_right['x'], golden_right['y_top']

    for fidx, frame in enumerate(trimmed_frames):
        debug_vis = frame.copy()
        if fidx in segs:
            if 1 in segs[fidx]:
                m1 = segs[fidx][1].astype(np.uint8)
                m1_thick = cv2.dilate(m1, np.ones((7, 7), np.uint8), iterations=2)
                debug_vis[m1_thick > 0] = debug_vis[m1_thick > 0] * 0.2 + np.array([0, 0, 255]) * 0.8

                contours, _ = cv2.findContours(m1_thick, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(debug_vis, contours, -1, (255, 255, 255), 2)

                if (r := extract_line_from_mask(segs[fidx][1])): lx, lyt = r

            if 2 in segs[fidx]:
                m2 = segs[fidx][2].astype(np.uint8)
                m2_thick = cv2.dilate(m2, np.ones((7, 7), np.uint8), iterations=2)
                debug_vis[m2_thick > 0] = debug_vis[m2_thick > 0] * 0.2 + np.array([0, 0, 255]) * 0.8

                contours, _ = cv2.findContours(m2_thick, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(debug_vis, contours, -1, (255, 255, 255), 2)

                if (r := extract_line_from_mask(segs[fidx][2])): rx, ryt = r
        debug_frames.append(debug_vis)

        frame_copy = frame.copy()
        frame_copy[int(h * PITCH_BOTTOM_MASK_FRAC):h, :] = 0
        scale = TARGET_PITCH_W / max(10, rx - lx)
        M = np.float32([[scale, 0, TARGET_LEFT_X - lx*scale],
                         [0,     scale, TARGET_LINE_Y - lyt*scale]])
        out = cv2.warpAffine(frame_copy, M, (TARGET_NORM_W, TARGET_NORM_H),
                             borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0))
        out[:, 0:PITCH_EDGE_MARGIN] = 0
        out[:, TARGET_NORM_W - PITCH_EDGE_MARGIN:TARGET_NORM_W] = 0
        stab_frames.append(out)

    return stab_frames, debug_frames



### 7e. Run it

Loads the models this stage needs (YOLO x2 for trimming, Grounding DINO + SAM 2 for crease tracking), runs trimming + crease detection + stabilization, then frees those models before the next stage loads its own.


In [12]:
print("Loading YOLO (x2) + Grounding DINO + SAM 2...")
yolo_obj = YOLO(YOLO_OBJ_WEIGHTS).to("cuda")
yolo_pose = YOLO(YOLO_POSE_WEIGHTS).to("cuda")
dino_processor = AutoProcessor.from_pretrained(DINO_MODEL_NAME)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_MODEL_NAME).to(device)
sam2 = build_sam2_video_predictor(SAM2_CONFIG, SAM2_WEIGHTS, device=device)

total_t = time.time()
timings = {}

print("--- Stage 1: Load & Trim ---")
t = time.time()
frames, fps = load_frames(input_video)
start_idx, end_idx, box_history = trim_video(frames, yolo_pose, yolo_obj)
if start_idx is not None:
    frames = frames[start_idx:end_idx+1]
if VISUALIZATION:
    h_t, w_t = frames[0].shape[:2]
    tw = cv2.VideoWriter(os.path.join(out_dir, "yolo_trimmed.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w_t, h_t))
    for f in frames: tw.write(f)
    tw.release()
print("Stage 1 done.")

print("--- Stage 2: Pitch Stabilization ---")
stabilized_frames, debug_pitch_frames = stabilize(frames, dino_processor, dino_model, sam2, device)
rgb_path = os.path.join(out_dir, "stabilized_rgb.mp4")
out_h, out_w = stabilized_frames[0].shape[:2]
writer = cv2.VideoWriter(rgb_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (out_w, out_h))
for f in stabilized_frames: writer.write(f)
writer.release()
if VISUALIZATION and debug_pitch_frames:
    dh, dw = debug_pitch_frames[0].shape[:2]
    dw_v = cv2.VideoWriter(os.path.join(out_dir, "pitch_detected.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), fps, (dw, dh))
    for f in debug_pitch_frames: dw_v.write(f)
    dw_v.release()
del frames; gc.collect(); torch.cuda.empty_cache()
print("Stage 2 done.")

print("Freeing crease-detection models...")
del yolo_obj, yolo_pose, dino_processor, dino_model, sam2
gc.collect()
torch.cuda.empty_cache()


Loading YOLO (x2) + Grounding DINO + SAM 2...
--- Stage 1: Load & Trim ---
Stage 1 done.
--- Stage 2: Pitch Stabilization ---


propagate in video: 100%|██████████| 46/46 [00:01<00:00, 27.47it/s]


Stage 2 done.
Freeing crease-detection models...


### 7f. Output of Crease line detection


In [13]:
import subprocess
import os
from IPython.display import Video, display

# 2. Silently convert and display the video at a smaller size
input_mp4 = os.path.join(out_dir, "yolo_trimmed.mp4")
play_mp4 = os.path.join(out_dir, "yolo_trimmed_play.mp4")

if os.path.exists(input_mp4):
    cmd = f'ffmpeg -y -i "{input_mp4}" -vcodec libx264 -f mp4 "{play_mp4}" -loglevel quiet'
    subprocess.run(cmd, shell=True)

    if os.path.exists(play_mp4):
        display(Video(play_mp4, embed=True, width=480))


In [14]:
import subprocess
import os
from IPython.display import Video, display

# 1. Print frame count
print(f"Stabilized frames: {len(stabilized_frames)}")

# 2. Paths
stabilized_mp4 = os.path.join(out_dir, "stabilized_rgb.mp4")
stabilized_play = os.path.join(out_dir, "stabilized_rgb_play.mp4")
pitch_mp4 = os.path.join(out_dir, "pitch_detected.mp4")
pitch_play = os.path.join(out_dir, "pitch_detected_play.mp4")

# 3. Convert and display Pitch Detection overlay video (first step)
if os.path.exists(pitch_mp4):
    cmd = f'ffmpeg -y -i "{pitch_mp4}" -vcodec libx264 -f mp4 "{pitch_play}" -loglevel quiet'
    subprocess.run(cmd, shell=True)
    if os.path.exists(pitch_play):
        print("Pitch Crease Detection:")
        display(Video(pitch_play, embed=True, width=480))


Stabilized frames: 46
Pitch Crease Detection:


## 8. Video stabilization

Writes and displays the warped `stabilized_frames` computed in the previous cell. Camera pan/zoom is now removed: every frame's crease lines sit on the same fixed pixels. Nothing is recomputed here.


### Show the output of video stabilization


In [15]:
import subprocess
import os
from IPython.display import Video, display

stabilized_mp4 = os.path.join(out_dir, "stabilized_rgb.mp4")
stabilized_play = os.path.join(out_dir, "stabilized_rgb_play.mp4")

# 4. Convert and display Stabilized video (final step)
if os.path.exists(stabilized_mp4):
    cmd = f'ffmpeg -y -i "{stabilized_mp4}" -vcodec libx264 -f mp4 "{stabilized_play}" -loglevel quiet'
    subprocess.run(cmd, shell=True)
    if os.path.exists(stabilized_play):
        print("Stabilized Video:")
        display(Video(stabilized_play, embed=True, width=480))


Stabilized Video:


## 9. Segmentation (SAM 3.1)

Isolates the cricket bat from the stabilized video using SAM 3.1. Two sequential text-prompted sessions ('bat' and 'wicket') run so the wicket mask can be subtracted from the bat mask, removing noise blobs and keeping one clean tracked bat mask per frame.


### 9a. Run one SAM 3.1 text-prompted tracking session

The core session runner: starts a SAM session on the frames, logs Grounding-DINO's raw detections at a low threshold for debugging, seeds tracking on the best frame, and propagates masks through the whole clip, with a fallback retry if the tracker drops its only detection.


In [16]:
import sys
import numpy as np
import cv2
import torch
import decord



# ── CONFIG (local aliases for readability) ──────────────────────────────────
BAT_PROMPT = SAM_BAT_PROMPT
WICKET_PROMPT = SAM_WICKET_PROMPT
MIN_MASK_AREA = SAM_MIN_MASK_AREA
GDINO_DETECTION_THRESHOLD = SAM_GDINO_THRESHOLD


# ── SAM SESSION ───────────────────────────────────────────────────────────────

def run_session(predictor, pil_frames, text_prompt, frame_index):
    """Run one SAM text-prompted session over the whole video.

    pil_frames : list of PIL.Image objects (RGB) — the same frames that
                 find_highest_confidence_frame scanned. Passing PIL images
                 directly into SAM3 (supported by io_utils.py line 44) ensures
                 both the scan pass and tracker-init pass see IDENTICAL pixels,
                 eliminating the decord vs. OpenCV decode mismatch that caused
                 frame scores to collapse from ~0.24 to 0.0 in SAM3's session.

    Returns {frame_idx: outputs}.
    """
    # Ensure predictor is initialized with the configured GDINO threshold
    if hasattr(predictor, "score_threshold_detection"):
        predictor.score_threshold_detection = GDINO_DETECTION_THRESHOLD
    if hasattr(predictor.model, "score_threshold_detection"):
        predictor.model.score_threshold_detection = GDINO_DETECTION_THRESHOLD

    # Safely monkey-patch to ignore offload_state_to_cpu without creating an infinite closure chain
    if not hasattr(predictor.model, "_orig_init_state"):
        predictor.model._orig_init_state = predictor.model.init_state
        def _compat(*a, **kw):
            kw.pop("offload_state_to_cpu", None)
            return predictor.model._orig_init_state(*a, **kw)
        predictor.model.init_state = _compat

    # Pass PIL image list directly — SAM3's io_utils.load_resource_as_video_frames
    # accepts a list of PIL Images as resource_path (see io_utils.py line 44-69).
    resp = predictor.handle_request(request=dict(type="start_session", resource_path=pil_frames))
    session_id = resp["session_id"]

    try:
        # Run raw GDINO detection with a very low threshold to log all detections (above and below default threshold)
        old_thresh = getattr(predictor, "score_threshold_detection", GDINO_DETECTION_THRESHOLD)
        if hasattr(predictor, "score_threshold_detection"):
            predictor.score_threshold_detection = SAM_GDINO_RAW_THRESHOLD
        if hasattr(predictor.model, "score_threshold_detection"):
            predictor.model.score_threshold_detection = SAM_GDINO_RAW_THRESHOLD

        tmp_resp = predictor.handle_request(request=dict(type="start_session", resource_path=pil_frames))
        tmp_sid = tmp_resp["session_id"]
        try:
            resp_all = predictor.handle_request(request=dict(
                type="add_prompt", session_id=tmp_sid,
                frame_index=frame_index, text=text_prompt, obj_id=1,
            ))
            all_obj_ids = resp_all["outputs"].get("out_obj_ids", [])
            all_probs = resp_all["outputs"].get("out_probs", [])
            all_boxes = resp_all["outputs"].get("out_boxes_xywh", [])

            print(f"    [sam] raw GDINO detections (threshold={SAM_GDINO_RAW_THRESHOLD}) for '{text_prompt}' on frame {frame_index}:")
            if len(all_obj_ids) == 0:
                print("      - No detections found even at 0.1 threshold.")
            for obj_id, prob, box in zip(all_obj_ids, all_probs, all_boxes):
                status = "PASS" if prob >= GDINO_DETECTION_THRESHOLD else "BELOW_THRESHOLD"
                print(f"      - Object {obj_id}: confidence {prob:.4f} (Status: {status}, box: {box})")
        except Exception as raw_det_err:
            print(f"    WARNING: [sam] error fetching raw detections: {raw_det_err}")
        finally:
            try:
                predictor.handle_request(request=dict(type="reset_session", session_id=tmp_sid))
                predictor.handle_request(request=dict(type="close_session", session_id=tmp_sid))
            except Exception:
                pass
            if hasattr(predictor, "score_threshold_detection"):
                predictor.score_threshold_detection = old_thresh
            if hasattr(predictor.model, "score_threshold_detection"):
                predictor.model.score_threshold_detection = old_thresh

        # Now add the prompt for actual tracking/propagation using the normal threshold
        resp2 = predictor.handle_request(request=dict(
            type="add_prompt", session_id=session_id,
            frame_index=frame_index, text=text_prompt, obj_id=1,
        ))
        out_obj_ids = resp2["outputs"].get("out_obj_ids", [])
        out_probs = resp2["outputs"].get("out_probs", [])
        out_boxes = resp2["outputs"].get("out_boxes_xywh", [])
        n_det = len(out_obj_ids)
        print(f"    [sam] tracker init for '{text_prompt}' on frame {frame_index}: {n_det} detection(s) above threshold")
        for obj_id, prob, box in zip(out_obj_ids, out_probs, out_boxes):
            print(f"      - Object ID {obj_id} (DINO detector confidence: {prob:.4f}, box: {box})")

        if n_det == 0:
            # No detection — propagate directly. SAM3 returns empty masks on all
            # frames with no crash. This matches the original behaviour.
            print(f"    WARNING: [sam] DINO failed to find '{text_prompt}' on frame {frame_index}! tracking will search other frames.")
            outputs = {}
            for r in predictor.handle_stream_request(request=dict(type="propagate_in_video", session_id=session_id)):
                frame_idx = r["frame_index"]
                out_dict = r["outputs"]
                outputs[frame_idx] = out_dict


            return outputs

        # n_det >= 1: a detection was found. Propagate directly.
        try:
            outputs = {}
            for r in predictor.handle_stream_request(request=dict(type="propagate_in_video", session_id=session_id)):
                frame_idx = r["frame_index"]
                out_dict = r["outputs"]
                outputs[frame_idx] = out_dict


            return outputs
        except RuntimeError as e:
            if "No points are provided" in str(e):
                # The Grounding Detector found n_det=1 but the Tracker failed to
                # initialise from it — session tracker state is now corrupted.
                #
                # Correct fix: reset, then re-add the REAL text prompt on frame 0.
                print(f"    WARNING: [sam] '{text_prompt}' tracker dropped detection. retrying with frame-0 seed.")
                try:
                    predictor.handle_request(request=dict(type="reset_session", session_id=session_id))
                    predictor.handle_request(request=dict(
                        type="add_prompt", session_id=session_id,
                        frame_index=0, text=text_prompt, obj_id=1,
                    ))
                    outputs = {}
                    for r in predictor.handle_stream_request(request=dict(type="propagate_in_video", session_id=session_id)):
                        frame_idx = r["frame_index"]
                        out_dict = r["outputs"]
                        outputs[frame_idx] = out_dict


                    return outputs
                except Exception as fallback_err:
                    print(f"    WARNING: [sam] frame-0 fallback failed ({fallback_err}). returning empty.")
                    return {}
            else:
                raise

    finally:
        # Always clean up — even if propagation throws mid-way.
        # Without this, the session's GPU frame embeddings stay live on the
        # predictor, leaking ~8 GB VRAM per failed video.
        try:
            predictor.handle_request(request=dict(type="reset_session", session_id=session_id))
        except Exception:
            pass
        try:
            predictor.handle_request(request=dict(type="close_session", session_id=session_id))
        except Exception:
            pass


# ── MASK HELPERS ──────────────────────────────────────────────────────────────

def merged_mask(out_dict, H, W):
    """Union of all binary masks for one frame → bool HxW."""
    result = np.zeros((H, W), dtype=bool)
    for m in out_dict.get("out_binary_masks", []):
        m = np.asarray(m)
        if m.shape != (H, W):
            m = cv2.resize(m.astype(np.float32), (W, H), interpolation=cv2.INTER_NEAREST)
        result |= (m > 0.5)
    return result


def remove_small_blobs(mask, min_area):
    """Remove connected components whose pixel area < min_area."""
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    cleaned = np.zeros_like(mask)
    for lbl in range(1, n_labels):
        if stats[lbl, cv2.CC_STAT_AREA] >= min_area:
            cleaned |= (labels == lbl)
    return cleaned


def compute_wicket_zone(wkt_masks, all_fidxs, pad=SAM_WICKET_ZONE_PAD):
    """Average bounding rectangle of the wicket across detected frames, padded.
    Returns (x1, y1, x2, y2) floats, or None."""
    x1s, y1s, x2s, y2s = [], [], [], []
    for f in all_fidxs:
        m = wkt_masks[f]
        if not m.any():
            continue
        ys, xs = np.where(m)
        x1s.append(xs.min()); y1s.append(ys.min())
        x2s.append(xs.max()); y2s.append(ys.max())
    if not x1s:
        return None
    cx = (np.mean(x1s) + np.mean(x2s)) / 2
    cy = (np.mean(y1s) + np.mean(y2s)) / 2
    hw = (np.mean(x2s) - np.mean(x1s)) / 2 * pad
    hh = (np.mean(y2s) - np.mean(y1s)) / 2 * pad
    return (cx - hw, cy - hh, cx + hw, cy + hh)


def blob_in_wicket_zone(blob_mask, zone, overlap_thresh=SAM_WICKET_OVERLAP_THRESH):
    """True if more than overlap_thresh of the blob lies inside the wicket zone."""
    if zone is None:
        return False
    H, W = blob_mask.shape
    zx1, zy1 = max(0, int(zone[0])), max(0, int(zone[1]))
    zx2, zy2 = min(W, int(zone[2])), min(H, int(zone[3]))
    blob_area = int(blob_mask.sum())
    if blob_area == 0:
        return False
    overlap = int(blob_mask[zy1:zy2, zx1:zx2].sum())
    return (overlap / blob_area) > overlap_thresh


def select_tracked_bat_masks(filtered_masks, wkt_zone, all_fidxs):
    """Per-frame, keep the single connected component that best represents the bat.

    Pass 1 — reference bat area from blobs clearly outside the wicket zone.
    Pass 2 — per frame skip blobs that are BOTH inside the wicket zone AND
    smaller than 40% of the reference (wicket artifacts); keep the best remaining
    blob, using previous-frame centroid as a continuity tiebreaker.
    """
    ref_areas = []
    for f in all_fidxs:
        mask = filtered_masks[f]
        if not mask.any():
            continue
        n_lbl, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
        for lbl in range(1, n_lbl):
            if not blob_in_wicket_zone((labels == lbl), wkt_zone):
                ref_areas.append(stats[lbl, cv2.CC_STAT_AREA])

    ref_bat_area = float(np.median(ref_areas)) if ref_areas else 0.0
    min_keep_area = ref_bat_area * SAM_REF_AREA_MIN_KEEP
    print(f"  [sam] bat reference area: {ref_bat_area:.0f}px | wicket-zone blobs < {min_keep_area:.0f}px excluded")

    tracked = {}
    prev_centroid = None
    for fidx in all_fidxs:
        mask = filtered_masks[fidx]
        if not mask.any():
            tracked[fidx] = np.zeros_like(mask)
            continue

        n_lbl, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
        candidates = []
        for lbl in range(1, n_lbl):
            area = stats[lbl, cv2.CC_STAT_AREA]
            blob = (labels == lbl)
            if blob_in_wicket_zone(blob, wkt_zone) and area < min_keep_area:
                continue
            cx, cy = centroids[lbl]
            candidates.append((lbl, area, cx, cy))

        if not candidates:
            tracked[fidx] = np.zeros_like(mask)
            continue

        if prev_centroid is None or len(candidates) == 1:
            best = max(candidates, key=lambda x: x[1])
        else:
            px, py = prev_centroid
            best = min(candidates, key=lambda x: np.hypot(x[2] - px, x[3] - py) / (x[1] + 1))

        lbl, _, cx, cy = best
        tracked[fidx] = (labels == lbl)
        prev_centroid = (cx, cy)
    return tracked
def find_highest_confidence_frame(sam, pil_frames, text_prompt, default_frame):
    import torch
    import numpy as np

    if sam is None:
        return default_frame

    # Safely monkey-patch to ignore offload_state_to_cpu without creating an infinite closure chain
    if not hasattr(sam.model, "_orig_init_state"):
        sam.model._orig_init_state = sam.model.init_state
        def _compat(*a, **kw):
            kw.pop("offload_state_to_cpu", None)
            return sam.model._orig_init_state(*a, **kw)
        sam.model.init_state = _compat

    # Store old threshold to restore later
    old_thresh = getattr(sam, "score_threshold_detection", SAM_GDINO_THRESHOLD)
    if hasattr(sam, "score_threshold_detection"):
        sam.score_threshold_detection = SAM_GDINO_SCAN_THRESHOLD
    if hasattr(sam.model, "score_threshold_detection"):
        sam.model.score_threshold_detection = SAM_GDINO_SCAN_THRESHOLD

    # Start a temporary session on the PIL frames
    resp = sam.handle_request(request=dict(type="start_session", resource_path=pil_frames))
    session_id = resp["session_id"]

    best_frame = default_frame
    best_score = -1.0

    try:
        # Loop through frames to query SAM3's internal Grounding DINO detector
        for idx in range(len(pil_frames)):
            prompt_resp = sam.handle_request(request=dict(
                type="add_prompt",
                session_id=session_id,
                frame_index=idx,
                text=text_prompt,
                obj_id=1,
            ))
            out_probs = prompt_resp["outputs"].get("out_probs", [])
            if len(out_probs) > 0:
                max_score = float(np.max(out_probs))
                if max_score > best_score:
                    best_score = max_score
                    best_frame = idx
    except Exception as e:
        print(f"    WARNING: [sam] error during DINO scanning: {e}")
    finally:
        # Close temp session and restore threshold
        try:
            sam.handle_request(request=dict(type="reset_session", session_id=session_id))
            sam.handle_request(request=dict(type="close_session", session_id=session_id))
        except Exception:
            pass
        if hasattr(sam, "score_threshold_detection"):
            sam.score_threshold_detection = old_thresh
        if hasattr(sam.model, "score_threshold_detection"):
            sam.model.score_threshold_detection = old_thresh

    if best_score < GDINO_DETECTION_THRESHOLD:
        print(f"    [sam] highest confidence for '{text_prompt}' too low ({best_score:.4f} < {GDINO_DETECTION_THRESHOLD}). falling back to default frame {default_frame}.")
        return default_frame

    print(f"    [sam] selected frame {best_frame} for '{text_prompt}' with highest confidence {best_score:.4f} (default: {default_frame})")
    return best_frame


# ── ENTRY POINT ───────────────────────────────────────────────────────────────

def segment(sam, video_path, dino_model=None, dino_processor=None, save_debug=False):
    """Segment the tracked bat on the anchored RGB video.

    save_debug=True writes bat/wicket/final mask overlay videos to the output
    directory. Off by default — enable via --visualization ON in main.py.

    Returns (tracked_masks: {fidx: bool HxW}, rgb_frames: list[RGB uint8], sam_stats).
    """
    import gc
    from PIL import Image as _PIL_Image
    vr = decord.VideoReader(video_path)
    n_fr = len(vr)
    frames = [vr[i].asnumpy() for i in range(n_fr)]   # RGB uint8 numpy, kept for downstream
    H, W = frames[0].shape[:2]
    # Explicitly delete the VideoReader to release its GPU/CPU decode buffers immediately.
    del vr
    gc.collect()
    print(f"  [sam] processing {n_fr} frames at {W}x{H} resolution")

    # Convert numpy frames to PIL Images ONCE.
    # Both the DINO scan and SAM3 tracker-init will use these same PIL objects,
    # guaranteeing identical pixel values in both passes and eliminating the
    # decord vs. OpenCV decode mismatch that caused tracker scores to drop to 0.
    pil_frames = [_PIL_Image.fromarray(f) for f in frames]

    default_bat_frame = n_fr // 2
    default_wicket_frame = n_fr // 2

    actual_bat_frame = find_highest_confidence_frame(sam, pil_frames, BAT_PROMPT, default_bat_frame)
    actual_wicket_frame = find_highest_confidence_frame(sam, pil_frames, WICKET_PROMPT, default_wicket_frame)

    print("  [sam] running session 1: bat")
    bat_out = run_session(sam, pil_frames, BAT_PROMPT, actual_bat_frame)
    print("  [sam] running session 2: wicket")
    wkt_out = run_session(sam, pil_frames, WICKET_PROMPT, actual_wicket_frame)

    empty = np.zeros((H, W), dtype=bool)
    all_fidxs = sorted(set(bat_out) | set(wkt_out))
    bat_masks = {f: merged_mask(bat_out[f], H, W) if f in bat_out else empty.copy() for f in all_fidxs}
    wkt_masks = {f: merged_mask(wkt_out[f], H, W) if f in wkt_out else empty.copy() for f in all_fidxs}

    num_bat_raw = sum(1 for m in bat_masks.values() if m.any())
    num_wkt_raw = sum(1 for m in wkt_masks.values() if m.any())

    # Free the large raw session output dicts now that masks are extracted.
    del bat_out, wkt_out
    gc.collect()
    torch.cuda.empty_cache()

    clean_masks = {f: bat_masks[f] & ~wkt_masks[f] for f in all_fidxs}
    filtered_masks = {f: remove_small_blobs(clean_masks[f], MIN_MASK_AREA) for f in all_fidxs}

    wkt_zone = compute_wicket_zone(wkt_masks, all_fidxs)
    tracked_masks = select_tracked_bat_masks(filtered_masks, wkt_zone, all_fidxs)

    # SAM mask debug videos — only written when visualization mode is on
    if save_debug:
        import os
        import cv2
        out_dir  = os.path.dirname(video_path)
        out_name = os.path.basename(out_dir)
        fps      = SAM_DEBUG_FPS
        fourcc   = cv2.VideoWriter_fourcc(*"mp4v")

        vw_bat   = cv2.VideoWriter(os.path.join(out_dir, f"{out_name}_sam_mask_bat.mp4"),    fourcc, fps, (W, H))
        vw_wkt   = cv2.VideoWriter(os.path.join(out_dir, f"{out_name}_sam_mask_wicket.mp4"), fourcc, fps, (W, H))
        vw_final = cv2.VideoWriter(os.path.join(out_dir, f"{out_name}_sam_mask_final.mp4"),  fourcc, fps, (W, H))

        for f in range(n_fr):
            frame_f = frames[f].astype(np.float32)
            bat_m   = bat_masks.get(f, empty)
            wkt_m   = wkt_masks.get(f, empty)
            final_m = tracked_masks.get(f, empty)

            img_bat = frame_f.copy()
            img_bat[bat_m] = img_bat[bat_m] * 0.5 + np.array([255, 0, 0], dtype=np.float32) * 0.5
            img_bat_bgr = cv2.cvtColor(img_bat.clip(0, 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
            cv2.putText(img_bat_bgr, f"f{f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 3)
            cv2.putText(img_bat_bgr, f"f{f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
            vw_bat.write(img_bat_bgr)

            img_wkt = frame_f.copy()
            img_wkt[wkt_m] = img_wkt[wkt_m] * 0.5 + np.array([255, 0, 0], dtype=np.float32) * 0.5
            img_wkt_bgr = cv2.cvtColor(img_wkt.clip(0, 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
            cv2.putText(img_wkt_bgr, f"f{f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 3)
            cv2.putText(img_wkt_bgr, f"f{f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
            vw_wkt.write(img_wkt_bgr)

            img_final = frame_f.copy()
            img_final[final_m] = img_final[final_m] * 0.5 + np.array([0, 255, 0], dtype=np.float32) * 0.5
            img_final_bgr = cv2.cvtColor(img_final.clip(0, 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
            if wkt_zone:
                zx1, zy1, zx2, zy2 = map(int, wkt_zone)
                cv2.rectangle(img_final_bgr, (zx1, zy1), (zx2, zy2), (0, 255, 255), 2)
            cv2.putText(img_final_bgr, f"f{f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 3)
            cv2.putText(img_final_bgr, f"f{f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
            vw_final.write(img_final_bgr)

        vw_bat.release()
        vw_wkt.release()
        vw_final.release()
        print(f"  [debug] SAM mask videos saved to {out_dir}")

    trk_hit = sum(1 for m in tracked_masks.values() if m.any())
    print(f"  [sam] bat tracked in {trk_hit}/{len(all_fidxs)} frames")
    sam_stats = {
        "total_frames":   n_fr,
        "sam_frames":     len(all_fidxs),
        "tracked_frames": trk_hit,
        "bat_raw":        num_bat_raw,
        "wkt_raw":        num_wkt_raw,
    }
    return tracked_masks, frames, sam_stats



### 9b. Run it

Loads SAM 3.1 (the only model this stage needs: `dino_model`/`dino_processor` are accepted by `segment()` but unused, so they're not loaded here), runs the two tracking sessions under bfloat16 autocast, displays the final cleaned bat-mask video, then frees SAM 3.1 before the depth stage loads its own model.


In [17]:
print("Loading SAM 3.1...")
sam3 = build_sam3_multiplex_video_predictor(checkpoint_path=SAM3_WEIGHTS, use_fa3=False)

print("--- Stage 4: SAM Segmentation ---")
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    tracked_masks, rgb_frames, sam_stats = segment(sam3, rgb_path, save_debug=VISUALIZATION)
print("Stage 4 done.")

print("Freeing SAM 3.1...")
del sam3
gc.collect()
torch.cuda.empty_cache()


Loading SAM 3.1...
dynamic_multimask_via_stability is reset to False in the multiplex model
Missing keys: ['maskmem_tpos_enc', 'interactivity_no_mem_embed', 'no_obj_embed_spatial', 'output_valid_embed', 'output_invalid_embed', 'backbone.vision_backbone.trunk.pos_embed', 'backbone.vision_backbone.trunk.patch_embed.proj.weight', 'backbone.vision_backbone.trunk.blocks.0.norm1.weight', 'backbone.vision_backbone.trunk.blocks.0.norm1.bias', 'backbone.vision_backbone.trunk.blocks.0.attn.freqs_cis', 'backbone.vision_backbone.trunk.blocks.0.attn.qkv.weight', 'backbone.vision_backbone.trunk.blocks.0.attn.qkv.bias', 'backbone.vision_backbone.trunk.blocks.0.attn.proj.weight', 'backbone.vision_backbone.trunk.blocks.0.attn.proj.bias', 'backbone.vision_backbone.trunk.blocks.0.norm2.weight', 'backbone.vision_backbone.trunk.blocks.0.norm2.bias', 'backbone.vision_backbone.trunk.blocks.0.mlp.fc1.weight', 'backbone.vision_backbone.trunk.blocks.0.mlp.fc1.bias', 'backbone.vision_backbone.trunk.blocks.0.mlp.

propagate_in_video: 100%|██████████| 33/33 [00:02<00:00, 11.16it/s]


  [sam] running session 2: wicket
    [sam] raw GDINO detections (threshold=0.1) for 'wicket' on frame 23:
      - Object 0: confidence 0.6684 (Status: PASS, box: [      0.475     0.55234    0.028125     0.11172])
    [sam] tracker init for 'wicket' on frame 23: 1 detection(s) above threshold
      - Object ID 0 (DINO detector confidence: 0.6684, box: [      0.475     0.55234    0.028125     0.11172])


propagate_in_video: 100%|██████████| 23/23 [00:02<00:00,  8.15it/s]


  [sam] bat reference area: 3182px | wicket-zone blobs < 1273px excluded
  [debug] SAM mask videos saved to /media/user/hugeDrive2/str8-bat/adnan/Bat_plane_v2.3/notebook/bat_notebook_1/outputs
  [sam] bat tracked in 46/46 frames
Stage 4 done.
Freeing SAM 3.1...


In [18]:
import os
import subprocess
from IPython.display import HTML, display

# 1. Define paths
out_dir = os.path.dirname(rgb_path)
out_name = os.path.basename(out_dir)

video_paths = {
    "Bat Mask (Red)": os.path.join(out_dir, f"{out_name}_sam_mask_bat.mp4"),
    "Wicket Mask (Red)": os.path.join(out_dir, f"{out_name}_sam_mask_wicket.mp4"),
    "Final Tracked Bat (Green)": os.path.join(out_dir, f"{out_name}_sam_mask_final.mp4")
}

# 2. Convert to H.264 (silent ffmpeg) & collect relative paths
html_cols = []
for label, path in video_paths.items():
    if os.path.exists(path):
        # Define the converted play path
        play_path = path.replace(".mp4", "_play.mp4")

        # Convert to libx264 silently
        cmd = f'ffmpeg -y -i "{path}" -vcodec libx264 -f mp4 "{play_path}" -loglevel quiet'
        subprocess.run(cmd, shell=True)

        # Get relative path for Jupyter HTML source
        rel_play_path = os.path.relpath(play_path, start=os.getcwd())

        video_html = f"""
        <div style="flex: 1; text-align: center; padding: 5px;">
            <h4 style="color: #7B00D4; font-family: sans-serif; margin-bottom: 8px;">{label}</h4>
            <video src="{rel_play_path}" controls autoplay loop muted width="100%"
                   style="border-radius: 8px; border: 2px solid #333; box-shadow: 0 4px 10px rgba(0,0,0,0.5);">
            </video>
        </div>
        """
        html_cols.append(video_html)
    else:
        html_cols.append(f"""
        <div style="flex: 1; text-align: center; padding: 5px; color: #ff5555; font-family: sans-serif;">
            <h4>{label}</h4>
            <div style="border: 2px dashed #ff5555; padding: 40px; border-radius: 8px; background: #221111;">
                File not found:<br>{os.path.basename(path)}
            </div>
        </div>
        """)

# 3. Render side-by-side inside a clean container
container_html = f"""
<div style="display: flex; gap: 15px; justify-content: space-between; background-color: #111; padding: 15px; border-radius: 12px; margin-top: 15px;">
    {''.join(html_cols)}
</div>
"""

display(HTML(container_html))


## 10. Depth analysis

Runs Depth-Anything-V2 on the stabilized frames to get a per-pixel relative-depth map, later used to give the bat's 2D keypoints a Z coordinate.


### 10a. Run depth estimation

Passes the hardened frames through Depth-Anything-V2 in batches, converts its disparity output to relative depth (inverse depth), and zeroes out depth on the black padding pixels introduced by stabilization.


In [19]:
import cv2
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor


def extract_depth(frames, depth_model, batch_size=DEPTH_BATCH_SIZE):
    """Run Depth-Anything on the frames.
    We pass the full frames to avoid batching errors caused by variable-sized crops.
    """
    total_frames = len(frames)
    FH, FW = frames[0].shape[:2]

    # Convert frames to PIL Images for the pipeline
    def _prep(frame):
        return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    pil_inputs = [_prep(f) for f in frames]

    results_generator = depth_model(pil_inputs, batch_size=batch_size)
    results_list = list(tqdm(results_generator, total=len(pil_inputs),
                             desc="Extracting Raw Depth (Batched)"))

    depths = []
    for fi, raw_data in enumerate(results_list):
        raw_array = raw_data["predicted_depth"]
        raw_array = raw_array.squeeze().cpu().numpy() if torch.is_tensor(raw_array) else np.squeeze(raw_array)
        depth_map = cv2.resize(raw_array, (FW, FH), interpolation=cv2.INTER_NEAREST)

        # Convert disparity to relative depth (inverse depth)
        depth_map = 1.0 / (depth_map + DEPTH_EPSILON)

        # Enforce exactly 0 depth on purely black padding pixels
        black_mask = np.all(frames[fi] == 0, axis=-1)
        depth_map[black_mask] = 0.0
        depths.append(depth_map)

    return depths



In [20]:
print("Loading Depth-Anything-V2...")
depth_model = hf_pipeline(task="depth-estimation", model=DEPTH_MODEL_NAME, device=device_id, torch_dtype=torch.float32)

print("--- Stage 3: Depth Extraction ---")
filtered_frames = []
clahe = cv2.createCLAHE(clipLimit=DEBUG_CLAHE_CLIP_LIMIT, tileGridSize=DEBUG_CLAHE_TILE_SIZE)
for f in stabilized_frames:
    black_mask = np.all(f < DEBUG_BLACK_THRESH, axis=-1)
    hsv = cv2.cvtColor(f, cv2.COLOR_BGR2HSV).astype(np.float32)
    h, s, v = cv2.split(hsv)
    s = np.clip(s * DEBUG_SAT_MULTIPLIER, 0, 255)
    f_sat = cv2.cvtColor(cv2.merge((h, s, v)).astype(np.uint8), cv2.COLOR_HSV2BGR)
    lab = cv2.cvtColor(f_sat, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = clahe.apply(l)
    clahe_bgr = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)
    gaussian = cv2.GaussianBlur(clahe_bgr, (0, 0), DEBUG_UNSHARP_SIGMA)
    sharpened = cv2.addWeighted(clahe_bgr, DEBUG_UNSHARP_STRENGTH, gaussian, -(DEBUG_UNSHARP_STRENGTH - 1.0), 0)
    sharpened[black_mask] = [0, 0, 0]
    filtered_frames.append(sharpened)
with torch.autocast(device_type="cuda", enabled=False):
    norm_depth = extract_depth(filtered_frames, depth_model)
gc.collect(); torch.cuda.empty_cache()
print("Stage 3 done.")

print("Freeing Depth-Anything-V2...")
del depth_model
gc.collect()
torch.cuda.empty_cache()


Loading Depth-Anything-V2...
--- Stage 3: Depth Extraction ---


Extracting Raw Depth (Batched): 100%|██████████| 46/46 [00:00<00:00, 1408306.45it/s]


Stage 3 done.
Freeing Depth-Anything-V2...


### Output


In [21]:
import subprocess
import os
import cv2
import numpy as np
from IPython.display import Video, display

# 1. Print frame count
print(f"Depth frames processed: {len(norm_depth)}")

# 2. Paths
depth_mp4 = os.path.join(out_dir, "depth_colormap_preview.mp4")
depth_play = os.path.join(out_dir, "depth_colormap_preview_play.mp4")

# 3. Create a temporary colormapped depth video
if len(norm_depth) > 0:
    H, W = stabilized_frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(depth_mp4, fourcc, fps, (W, H))

    # Calculate global range for consistent coloring (clip outliers)
    all_vals = np.concatenate([d[d != 0].ravel() for d in norm_depth]).astype(np.float64)
    if all_vals.size > 0:
        d_min = float(np.percentile(all_vals, 1))
        d_max = float(np.percentile(all_vals, 99))
        span = d_max - d_min if d_max > d_min else 1.0
    else:
        d_min, span = 0.0, 1.0

    for d in norm_depth:
        # Scale to 0-255 & apply TURBO colormap
        d_u8 = np.clip(((d - d_min) / span * 255), 0, 255).astype(np.uint8)
        d_bgr = cv2.applyColorMap(d_u8, cv2.COLORMAP_TURBO)
        if d_bgr.shape[:2] != (H, W):
            d_bgr = cv2.resize(d_bgr, (W, H), interpolation=cv2.INTER_LINEAR)
        writer.write(d_bgr)
    writer.release()

# 4. Convert and display the video
if os.path.exists(depth_mp4):
    cmd = f'ffmpeg -y -i "{depth_mp4}" -vcodec libx264 -f mp4 "{depth_play}" -loglevel quiet'
    subprocess.run(cmd, shell=True)
    if os.path.exists(depth_play):
        display(Video(depth_play, embed=True, width=480))


Depth frames processed: 46


## 11. Bat keypoints (grip & tip)

Turns the SAM bat mask into two labeled points per frame: the handle (grip) and the blade tip. YOLOv8-Pose locates the batsman's wrists to find the grip position, then PCA on the bat mask finds its major axis: the mask end farther from the grip is the tip.


In [22]:
import numpy as np


# ── CONFIG ──────────────────────────────────────────────────────────────────
YOLO_CONF = KP_YOLO_CONF
KP_CONF = KP_WRIST_CONF
GRIP_FALLBACK_FRAMES = KP_GRIP_FALLBACK_FRAMES


# ── GRIP DETECTION ────────────────────────────────────────────────────────────

def detect_grip(yolo, frame):
    """Run YOLOv8-pose; return grip_xy (wrist midpoint of the largest person) or None."""
    results = yolo(frame, conf=YOLO_CONF, verbose=False)[0]
    if results.boxes is None or len(results.boxes) == 0:
        return None

    boxes = results.boxes.xyxy.cpu().numpy()
    kps_data = results.keypoints.data.cpu().numpy() if results.keypoints is not None else None

    best_grip, best_area = None, 0.0
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        area = (x2 - x1) * (y2 - y1)
        kps = kps_data[i] if kps_data is not None else None
        if kps is None:
            continue
        lw, rw = kps[9], kps[10]            # COCO-17 wrists
        lw_ok, rw_ok = lw[2] >= KP_CONF, rw[2] >= KP_CONF
        if lw_ok and rw_ok:
            gx, gy = float((lw[0] + rw[0]) / 2), float((lw[1] + rw[1]) / 2)
        elif lw_ok:
            gx, gy = float(lw[0]), float(lw[1])
        elif rw_ok:
            gx, gy = float(rw[0]), float(rw[1])
        else:
            continue
        if area > best_area:
            best_area, best_grip = area, (gx, gy)
    return best_grip


# ── PCA BLADE TIP ─────────────────────────────────────────────────────────────

def pca_blade_tip(bat_bool, grip_xy):
    """PCA-based blade-tip finder. Blade is the mask end FARTHER from grip_xy
    (falls back to width vote when grip is unavailable).
    Returns (cx_tip, cy_tip, blade_xy, handle_pt)."""
    if not bat_bool.any():
        return np.nan, np.nan, None, None

    ys, xs = np.where(bat_bool)
    if len(xs) < KP_MIN_MASK_PIXELS:
        cx_fb, cy_fb = float(xs.mean()), float(ys.max())
        return cx_fb, cy_fb, (cx_fb, cy_fb), None

    cx, cy = float(xs.mean()), float(ys.mean())
    pts = np.column_stack([xs.astype(np.float64) - cx, ys.astype(np.float64) - cy])
    cov = (pts.T @ pts) / len(pts)
    eigvals, eigvecs = np.linalg.eigh(cov)
    major = eigvecs[:, 1]

    proj = pts @ major
    min_p, max_p = float(proj.min()), float(proj.max())
    axis_len = max_p - min_p

    p_min = np.array([cx + min_p * major[0], cy + min_p * major[1]])
    p_max = np.array([cx + max_p * major[0], cy + max_p * major[1]])

    END_FRAC = KP_END_FRAC
    n_min = int((proj <= min_p + axis_len * END_FRAC).sum())
    n_max = int((proj >= max_p - axis_len * END_FRAC).sum())

    if grip_xy is not None:
        d_min = float(np.hypot(p_min[0] - grip_xy[0], p_min[1] - grip_xy[1]))
        d_max = float(np.hypot(p_max[0] - grip_xy[0], p_max[1] - grip_xy[1]))
        if abs(d_min - d_max) > KP_GRIP_DISTANCE_TIEBREAK:
            blade_is_min = d_min > d_max          # grip distance decisive
        else:
            blade_is_min = n_min >= n_max         # width tiebreaker
    else:
        blade_is_min = n_min >= n_max             # width-only fallback

    blade_pt, handle_pt_raw = (p_min, p_max) if blade_is_min else (p_max, p_min)

    # 1. Tip Logic (Top 15%)
    TIP_FRAC = KP_TIP_FRAC
    tip_sel = (proj <= min_p + axis_len * TIP_FRAC) if blade_is_min else (proj >= max_p - axis_len * TIP_FRAC)
    if tip_sel.sum() > 0:
        cx_tip = float(cx + pts[tip_sel, 0].mean())
        cy_tip = float(cy + pts[tip_sel, 1].mean())
    else:
        cx_tip, cy_tip = float(blade_pt[0]), float(blade_pt[1])

    # 2. Handle Logic (Bottom 15%)
    HANDLE_FRAC = KP_HANDLE_FRAC
    handle_sel = (proj >= max_p - axis_len * HANDLE_FRAC) if blade_is_min else (proj <= min_p + axis_len * HANDLE_FRAC)
    if handle_sel.sum() > 0:
        cx_handle = float(cx + pts[handle_sel, 0].mean())
        cy_handle = float(cy + pts[handle_sel, 1].mean())
        handle_pt = (cx_handle, cy_handle)
    else:
        handle_pt = tuple(handle_pt_raw)

    return cx_tip, cy_tip, (cx_tip, cy_tip), tuple(handle_pt)


# ── ENTRY POINT ───────────────────────────────────────────────────────────────

def extract_xy(yolo_pose, tracked_masks, rgb_frames):
    """Per-frame grip + PCA blade tip. Returns (tip_xy, centroid_xy, handle_xy),
    each an (N,3) array of [frame_idx, x, y]."""
    total_fr = len(rgb_frames)
    H, W = rgb_frames[0].shape[:2]

    detected = {}
    last_grip, stale_grip = None, 0

    for fidx in range(total_fr):
        frame = rgb_frames[fidx]
        bat_bool = tracked_masks.get(fidx)
        if bat_bool is None:
            bat_bool = np.zeros((H, W), dtype=bool)

        grip_xy = detect_grip(yolo_pose, frame)
        if grip_xy is not None:
            last_grip, stale_grip = grip_xy, 0
        else:
            stale_grip += 1
            if stale_grip <= GRIP_FALLBACK_FRAMES and last_grip is not None:
                grip_xy = last_grip

        cx_tip, cy_tip, _, handle_pt = pca_blade_tip(bat_bool, grip_xy)

        ys_m, xs_m = np.where(bat_bool)
        if len(xs_m) > 0:
            cx_cen, cy_cen = float(xs_m.mean()), float(ys_m.mean())
        else:
            cx_cen, cy_cen = np.nan, np.nan

        if handle_pt is not None and not np.isnan(handle_pt[0]):
            cx_handle, cy_handle = float(handle_pt[0]), float(handle_pt[1])
        else:
            cx_handle, cy_handle = np.nan, np.nan

        detected[fidx] = (cx_cen, cy_cen, cx_tip, cy_tip, cx_handle, cy_handle)

    det_frames = np.array(sorted(detected.keys()), dtype=int)
    cols = lambda k: np.array([detected[f][k] for f in det_frames], dtype=float)
    tip_xy = np.column_stack([det_frames, cols(2), cols(3)])
    centroid_xy = np.column_stack([det_frames, cols(0), cols(1)])
    handle_xy = np.column_stack([det_frames, cols(4), cols(5)])

    valid = int((~np.isnan(cols(2))).sum())
    print(f"  bat detected in {valid}/{total_fr} frames")
    return tip_xy, centroid_xy, handle_xy



In [23]:
print("Loading YOLO-Pose...")
yolo_pose = YOLO(YOLO_POSE_WEIGHTS).to("cuda")

print("--- Stage 5: Keypoints ---")
with torch.autocast(device_type="cuda", enabled=False):
    tip_xy, _centroid_xy, handle_xy = extract_xy(yolo_pose, tracked_masks, rgb_frames)
print("Stage 5 done.")

if VISUALIZATION:
    print("--- Stage 6: Debug Videos ---")
    try:
        _save_debug_videos(tracked_masks, rgb_frames, norm_depth, tip_xy, handle_xy, out_dir, fps=fps)
    except Exception as e:
        print("Debug videos error:", e)
del tracked_masks, rgb_frames; gc.collect()

print("Freeing YOLO-Pose...")
del yolo_pose
gc.collect()
torch.cuda.empty_cache()


Loading YOLO-Pose...
--- Stage 5: Keypoints ---
  bat detected in 46/46 frames
Stage 5 done.
--- Stage 6: Debug Videos ---
  [debug] depth viz range [bat]: 0.00212034 → 0.00973962
  [debug] writing 46 frames to /media/user/hugeDrive2/str8-bat/adnan/Bat_plane_v2.3/notebook/bat_notebook_1/outputs
Debug videos error: name 'sample_depth' is not defined
Freeing YOLO-Pose...


In [24]:
import subprocess
import os
from IPython.display import Video, display

# Paths
input_mp4 = os.path.join(out_dir, "bat_mask_keypoints.mp4")
play_mp4 = os.path.join(out_dir, "bat_mask_keypoints_play.mp4")

# Silently convert and display
if os.path.exists(input_mp4):
    cmd = f'ffmpeg -y -i "{input_mp4}" -vcodec libx264 -f mp4 "{play_mp4}" -loglevel quiet'
    subprocess.run(cmd, shell=True)

    if os.path.exists(play_mp4):
        # Pass the clean relative path directly (no query parameters)
        rel_path = os.path.relpath(play_mp4, start=os.getcwd())
        display(Video(rel_path, embed=True, width=540))
else:
    print("❌ bat_mask_keypoints.mp4 not found. Make sure Stage 6 ran successfully.")


## 12. Final 3D trajectory

Fuses the 2D tip/handle keypoints with the depth maps into a full 3D trajectory: samples depth at each keypoint, fills gaps with a 9-state RTS Kalman smoother, smooths and upsamples the path, then renders an animated swept-plane visualization of the bat from multiple angles.


### 12a. Kalman gap-filling

A 9-state constant-acceleration model `[x, y, z, vx, vy, vz, ax, ay, az]`. An RTS (Rauch-Tung-Striebel) smoother fills any frame where the tip or handle wasn't detected, using the surrounding trajectory to infer where it must have been.


In [25]:
import os
import tempfile
import shutil
import subprocess
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection  # noqa: F401
from matplotlib.ticker import MultipleLocator
from matplotlib.colors import LinearSegmentedColormap
from scipy.interpolate import splprep, splev
from pykalman import KalmanFilter



# ── CONFIG (local aliases) ───────────────────────────────────────────────
SMOOTH_WINDOW_X = TRAJ_SMOOTH_WINDOW_X
SMOOTH_WINDOW_Y = TRAJ_SMOOTH_WINDOW_Y
SMOOTH_WINDOW_Z = TRAJ_SMOOTH_WINDOW_Z
DEPTH_PATCH_RADIUS = TRAJ_DEPTH_PATCH_RADIUS
Z_SCALE_FACTOR = TRAJ_Z_SCALE_FACTOR
CMAP_TIP, CMAP_HANDLE = TRAJ_CMAP_TIP, TRAJ_CMAP_HANDLE

RIBBON_ALPHA = TRAJ_RIBBON_ALPHA
_ribbon_cmap = LinearSegmentedColormap.from_list(
    "ribbon_yv", [TRAJ_RIBBON_COLOR_LOW, TRAJ_RIBBON_COLOR_HIGH]
)


# ── HELPERS ───────────────────────────────────────────────────────────────────

def smooth_1d(arr, w):
    if w <= 1 or len(arr) < w:
        return arr.copy()
    window = np.hanning(w)
    window /= window.sum()
    half_l = (w - 1) // 2
    half_r = w - 1 - half_l
    padded = np.pad(arr.astype(float), (half_l, half_r), mode="edge")
    return np.convolve(padded, window, mode="valid")

def catmull_rom_spline_1d(P, alpha=0.5, num_points=10):
    if len(P) < 4:
        return np.interp(np.linspace(0, len(P)-1, (len(P)-1)*num_points + 1), np.arange(len(P)), P)

    P_padded = np.pad(P, (1, 1), mode='edge')
    out = []

    for i in range(1, len(P_padded) - 2):
        p0, p1, p2, p3 = P_padded[i-1], P_padded[i], P_padded[i+1], P_padded[i+2]

        def get_t(t_val, a, p_0, p_1):
            a_val = (p_1 - p_0)**2
            b_val = a_val**0.5
            return (b_val**a) + t_val

        t0 = 0.0
        t1 = get_t(t0, alpha, p0, p1)
        t2 = get_t(t1, alpha, p1, p2)
        t3 = get_t(t2, alpha, p2, p3)

        if t1 == t0: t1 += 1e-6
        if t2 == t1: t2 += 1e-6
        if t3 == t2: t3 += 1e-6

        is_last = (i == len(P_padded) - 3)
        n_pts = num_points + 1 if is_last else num_points
        t = np.linspace(t1, t2, n_pts, endpoint=is_last)

        A1 = (t1-t)/(t1-t0)*p0 + (t-t0)/(t1-t0)*p1
        A2 = (t2-t)/(t2-t1)*p1 + (t-t1)/(t2-t1)*p2
        A3 = (t3-t)/(t3-t2)*p2 + (t-t2)/(t3-t2)*p3

        B1 = (t2-t)/(t2-t0)*A1 + (t-t0)/(t2-t0)*A2
        B2 = (t3-t)/(t3-t1)*A2 + (t-t1)/(t3-t1)*A3

        C = (t2-t)/(t2-t1)*B1 + (t-t1)/(t2-t1)*B2
        out.extend(C)

    return np.array(out)


def sample_depth(depth_map, cx, cy, radius):
    H, W = depth_map.shape[:2]
    px, py = int(round(cx)), int(round(cy))
    if radius == 0:
        return float(depth_map[np.clip(py, 0, H - 1), np.clip(px, 0, W - 1)])
    y0, y1 = max(0, py - radius), min(H, py + radius + 1)
    x0, x1 = max(0, px - radius), min(W, px + radius + 1)
    patch = depth_map[y0:y1, x0:x1]
    return float(patch.mean()) if patch.size > 0 else float(
        depth_map[np.clip(py, 0, H - 1), np.clip(px, 0, W - 1)])


# 9-state constant-acceleration model: [x, y, z, vx, vy, vz, ax, ay, az]
_F9 = np.array([
    [1, 0, 0, 1, 0, 0, 0.5, 0,   0  ],
    [0, 1, 0, 0, 1, 0, 0,   0.5, 0  ],
    [0, 0, 1, 0, 0, 1, 0,   0,   0.5],
    [0, 0, 0, 1, 0, 0, 1,   0,   0  ],
    [0, 0, 0, 0, 1, 0, 0,   1,   0  ],
    [0, 0, 0, 0, 0, 1, 0,   0,   1  ],
    [0, 0, 0, 0, 0, 0, 1,   0,   0  ],
    [0, 0, 0, 0, 0, 0, 0,   1,   0  ],
    [0, 0, 0, 0, 0, 0, 0,   0,   1  ],
], dtype=float)
_H9 = np.zeros((3, 9)); _H9[0, 0] = _H9[1, 1] = _H9[2, 2] = 1.0
_Q9 = np.diag(KALMAN_Q_DIAG)
_R9 = np.diag(KALMAN_R_DIAG)


def _rts_fill_3d(obs, label):
    valid = ~np.isnan(obs[:, 0])
    first = np.where(valid)[0][0]
    init_mean = np.array([obs[first, 0], obs[first, 1], obs[first, 2],
                          0., 0., 0., 0., 0., 0.])
    kf = KalmanFilter(transition_matrices=_F9, observation_matrices=_H9,
                      transition_covariance=_Q9, observation_covariance=_R9,
                      initial_state_mean=init_mean,
                      initial_state_covariance=np.eye(9) * KALMAN_INIT_COV_SCALE)
    smoothed, _ = kf.smooth(obs)
    result = obs.copy()
    gap_count = int((~valid).sum())
    result[~valid, 0:3] = smoothed[~valid, 0:3]
    print(f"  [3d] kalman: {label} gap-fill {gap_count} frames")
    return result, gap_count


def _interp_nans_3d(arr):
    for col in range(3):
        nans = np.isnan(arr[:, col])
        if nans.any() and not nans.all():
            x = np.arange(len(arr))
            arr[nans, col] = np.interp(x[nans], x[~nans], arr[~nans, col])
    return arr


def build_ribbon_mesh(x_tip, y_tip, z_tip,
                      x_handle, y_handle, z_handle,
                      up_to_frame):
    """Build a ruled-surface ribbon from frame 0 up to `up_to_frame` inclusive.

    Vertices: handle side (rows 0..n-1) + tip side (rows n..2n-1).
    Each quad is split into 2 triangles, depth-coloured via _ribbon_cmap.
    Returns (verts, faces, colors) or None if fewer than 2 frames.
    """
    n = up_to_frame + 1
    if n < 2:
        return None

    verts = np.empty((2 * n, 3), dtype=float)
    # mpl 3D axes layout: (X, Z_depth, Y_height)
    verts[:n, 0] = x_handle[:n];  verts[:n, 1] = z_handle[:n];  verts[:n, 2] = y_handle[:n]
    verts[n:, 0] = x_tip[:n];   verts[n:, 1] = z_tip[:n];   verts[n:, 2] = y_tip[:n]

    quad_z = (z_handle[:n-1] + z_handle[1:n] + z_tip[:n-1] + z_tip[1:n]) / 4.0
    zmin, zmax = quad_z.min(), quad_z.max()
    z_norm = (quad_z - zmin) / (zmax - zmin + 1e-9)

    n_tris = (n - 1) * 2
    faces  = np.empty((n_tris, 3), dtype=int)
    colors = np.empty((n_tris, 4), dtype=float)

    for i in range(n - 1):
        A, B, C, D = i, i + 1, n + i + 1, n + i
        rgba = (*_ribbon_cmap(z_norm[i])[:3], RIBBON_ALPHA)
        faces[i*2]     = [A, B, D];  colors[i*2]     = rgba
        faces[i*2 + 1] = [B, C, D];  colors[i*2 + 1] = rgba

    return verts, faces, colors


def _render_frame(args):
    frame_idx, elev, azim, data, tmp_dir = args
    (x_tip, y_tip, z_tip, x_handle, y_handle, z_handle,
     z_n_tip, z_n_handle,
     x_min, x_max, y_min, y_max, z_min, z_max, dx, dy, dz) = data

    p_idx = frame_idx * TRAJ_SPLINE_UPSAMPLE if TRAJ_SPLINE_UPSAMPLE > 1 else frame_idx

    fig = plt.figure(figsize=TRAJ_RENDER_FIG_SIZE, dpi=TRAJ_RENDER_DPI)
    fig.patch.set_facecolor(TRAJ_BG_COLOR)
    ax = fig.add_subplot(111, projection="3d")
    ax.set_facecolor(TRAJ_AXIS_BG_COLOR)

    # Growing ribbon (swept bat-face plane)
    ribbon_i = build_ribbon_mesh(x_tip, y_tip, z_tip,
                                 x_handle, y_handle, z_handle, p_idx)
    if ribbon_i is not None:
        verts_i, faces_i, colors_i = ribbon_i
        polys_i = [verts_i[face] for face in faces_i]
        rc = Poly3DCollection(polys_i, zsort="average", zorder=2)
        rc.set_facecolor(colors_i.tolist())
        rc.set_edgecolor("none")
        ax.add_collection3d(rc)

    # Leading dots + current blade line (no trailing history)
    cmap_tip_c    = plt.get_cmap(CMAP_TIP)
    cmap_handle_c = plt.get_cmap(CMAP_HANDLE)
    c_tip    = cmap_tip_c(z_n_tip[p_idx])
    c_handle = cmap_handle_c(z_n_handle[p_idx])

    ax.scatter([x_tip[p_idx]],  [z_tip[p_idx]],  [y_tip[p_idx]],
               color="white", s=50, edgecolors=c_tip,  zorder=5)
    ax.scatter([x_handle[p_idx]], [z_handle[p_idx]], [y_handle[p_idx]],
               color="white", s=50, edgecolors=c_handle, zorder=5)
    ax.plot([x_tip[p_idx], x_handle[p_idx]],
            [z_tip[p_idx], z_handle[p_idx]],
            [y_tip[p_idx], y_handle[p_idx]],
            color="white", lw=3.0, zorder=5)

    # Fixed bounding box + labels
    ax.set_xlim([x_min - dx*0.05, x_max + dx*0.05])
    ax.set_ylim([z_min - dz*0.05, z_max + dz*0.05])
    ax.set_zlim([y_min - dy*0.05, y_max + dy*0.05])
    ax.set_box_aspect((dx, dz, dy), zoom=TRAJ_BOX_ZOOM)

    # Remove excess margins so the 3D plot fills the window
    fig.subplots_adjust(
        left=TRAJ_MARGIN_LEFT,
        right=TRAJ_MARGIN_RIGHT,
        bottom=TRAJ_MARGIN_BOTTOM,
        top=TRAJ_MARGIN_TOP
    )
    ax.set_xlabel("Width (X)",  color="lightgrey", labelpad=10)
    ax.set_ylabel("Depth (Z)",  color="lightgrey", labelpad=10)
    ax.set_zlabel("Height (Y)", color="lightgrey", labelpad=10)
    ax.tick_params(colors="grey")
    ax.xaxis.set_major_locator(MultipleLocator(TRAJ_AXIS_TICK_MAJOR))
    ax.yaxis.set_major_locator(MultipleLocator(TRAJ_AXIS_TICK_MAJOR))
    ax.zaxis.set_major_locator(MultipleLocator(TRAJ_AXIS_TICK_MAJOR))
    ax.view_init(elev=elev, azim=azim)

    # Depth colorbar
    norm_cb = plt.Normalize(vmin=z_min, vmax=z_max)
    sm = plt.cm.ScalarMappable(cmap=_ribbon_cmap, norm=norm_cb)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.05)
    cbar.set_label("Depth (Z)", color="lightgrey", labelpad=10)
    cbar.ax.yaxis.set_tick_params(color="grey", labelcolor="grey")
    cbar.outline.set_edgecolor("grey")

    fig.savefig(os.path.join(tmp_dir, f"frame_{frame_idx:04d}.png"),
                facecolor=fig.get_facecolor())
    plt.close(fig)


def save_2d_projections(x_tip, y_tip, z_tip, x_handle, y_handle, z_handle, out_dir):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor("#0d0d0d")

    # Colors
    c_tip = "#00ffff"    # Cyan
    c_handle = "#ff8c00" # Orange

    titles = ["Width vs Height (XY)", "Width vs Depth (XZ)", "Depth vs Height (ZY)"]

    # Setup styles
    for ax, title in zip(axes, titles):
        ax.set_facecolor("#111111")
        ax.title.set_text(title)
        ax.title.set_color("white")
        ax.title.set_fontsize(12)
        ax.tick_params(colors="grey")
        ax.grid(True, color="#333333", linestyle="--")
        for spine in ax.spines.values():
            spine.set_color("#444444")

    # Subplot 1: XY (Width vs Height)
    axes[0].plot(x_tip, y_tip, color=c_tip, label="Tip", lw=2, marker='o', markersize=4)
    axes[0].plot(x_handle, y_handle, color=c_handle, label="Handle", lw=2, marker='o', markersize=4)
    axes[0].set_xlabel("Width (X)", color="lightgrey")
    axes[0].set_ylabel("Height (Y)", color="lightgrey")
    axes[0].legend(facecolor="#1e1e1e", edgecolor="#444444", labelcolor="white")

    # Subplot 2: XZ (Width vs Depth)
    axes[1].plot(x_tip, z_tip, color=c_tip, label="Tip", lw=2, marker='o', markersize=4)
    axes[1].plot(x_handle, z_handle, color=c_handle, label="Handle", lw=2, marker='o', markersize=4)
    axes[1].set_xlabel("Width (X)", color="lightgrey")
    axes[1].set_ylabel("Depth (Z)", color="lightgrey")
    axes[1].legend(facecolor="#1e1e1e", edgecolor="#444444", labelcolor="white")

    # Subplot 3: ZY (Depth vs Height)
    axes[2].plot(z_tip, y_tip, color=c_tip, label="Tip", lw=2, marker='o', markersize=4)
    axes[2].plot(z_handle, y_handle, color=c_handle, label="Handle", lw=2, marker='o', markersize=4)
    axes[2].set_xlabel("Depth (Z)", color="lightgrey")
    axes[2].set_ylabel("Height (Y)", color="lightgrey")
    axes[2].legend(facecolor="#1e1e1e", edgecolor="#444444", labelcolor="white")

    plt.tight_layout()
    out_img = os.path.join(out_dir, "trajectory_2d_projections.png")
    fig.savefig(out_img, facecolor=fig.get_facecolor(), dpi=200)
    plt.close(fig)
    print(f"  [3d] 2D projections saved: {os.path.basename(out_img)}")


# ── ENTRY POINT ───────────────────────────────────────────────────────────────

def build_trajectory(tip_xy, handle_xy, norm_depth_arrays, out_dir):
    """Build the 3D bat trajectory and export a plane-only animated MP4.

    tip_xy / handle_xy : (N,3) arrays of [frame_idx, x, y]
    norm_depth_arrays  : list of normalised depth maps indexed by frame
    Returns a dict and writes traj_3d_{label}.mp4 (e.g. traj_3d_front.mp4) into out_dir.
    """
    os.makedirs(out_dir, exist_ok=True)

    tip_raw  = np.asarray(tip_xy,    dtype=float)
    handle_raw = np.asarray(handle_xy, dtype=float)
    tip_raw  = tip_raw[ ~(np.isnan(tip_raw[:,  1]) | np.isnan(tip_raw[:,  2]))]
    handle_raw = handle_raw[~(np.isnan(handle_raw[:, 1]) | np.isnan(handle_raw[:, 2]))]

    tip_frames_raw  = tip_raw[:,  0].astype(int)
    handle_frames_raw = handle_raw[:, 0].astype(int)
    tip_frame_to_idx  = {f: i for i, f in enumerate(tip_frames_raw)}
    handle_frame_to_idx = {f: i for i, f in enumerate(handle_frames_raw)}
    depth_lookup = {i: arr for i, arr in enumerate(norm_depth_arrays)}

    # ── Phase 1: build 3D observations ───────────────────────────────────────
    frame_start = min(tip_frames_raw.min(), handle_frames_raw.min())
    frame_end   = max(tip_frames_raw.max(), handle_frames_raw.max())
    all_frame_ids = np.arange(frame_start, frame_end + 1)
    total = len(all_frame_ids)

    obs_tip  = np.full((total, 3), np.nan)
    obs_handle = np.full((total, 3), np.nan)

    all_fidx = sorted(set(tip_frames_raw.tolist()) | set(handle_frames_raw.tolist()))
    for fidx in all_fidx:
        if fidx not in depth_lookup:
            if fidx in tip_frame_to_idx:
                print(f"  [3d] frame {fidx:03d}: tip   - no depth map, skipping")
            if fidx in handle_frame_to_idx:
                print(f"  [3d] frame {fidx:03d}: handle - no depth map, skipping")
            continue
        dmap = depth_lookup[fidx]
        rel  = fidx - frame_start
        valid_pixels = dmap[(dmap != 0) & np.isfinite(dmap)]
        if valid_pixels.size > 0:
            _mu  = float(valid_pixels.astype(np.float64).mean())
            _sig = float(valid_pixels.astype(np.float64).std())
            z_lo = _mu - TRAJ_OUTLIER_SIGMA * _sig
            z_hi = _mu + TRAJ_OUTLIER_SIGMA * _sig
        else:
            _mu, _sig = 0.0, 0.0
            z_lo, z_hi = -float('inf'), float('inf')

        if fidx in tip_frame_to_idx:
            i = tip_frame_to_idx[fidx]
            z = sample_depth(dmap, tip_raw[i, 1], tip_raw[i, 2], DEPTH_PATCH_RADIUS)
            if z != 0 and z_lo <= z <= z_hi:
                obs_tip[rel] = [tip_raw[i, 1], tip_raw[i, 2], z]
            elif z == 0:
                print(f"  [3d] frame {fidx:03d}: tip   - invalid (depth=0, padding pixel)")
            else:
                print(f"  [3d] frame {fidx:03d}: tip   - invalid (outlier z={z:.4f}, range=[{z_lo:.4f}, {z_hi:.4f}])")

        if fidx in handle_frame_to_idx:
            i = handle_frame_to_idx[fidx]
            z = sample_depth(dmap, handle_raw[i, 1], handle_raw[i, 2], DEPTH_PATCH_RADIUS)
            if z != 0 and z_lo <= z <= z_hi:
                obs_handle[rel] = [handle_raw[i, 1], handle_raw[i, 2], z]
            elif z == 0:
                print(f"  [3d] frame {fidx:03d}: handle - invalid (depth=0, padding pixel)")
            else:
                print(f"  [3d] frame {fidx:03d}: handle - invalid (outlier z={z:.4f}, range=[{z_lo:.4f}, {z_hi:.4f}])")

    n_valid_tip    = int((~np.isnan(obs_tip[:,  0])).sum())
    n_valid_handle = int((~np.isnan(obs_handle[:, 0])).sum())
    print(f"  [3d] depth fusion: tip {n_valid_tip}/{total} valid | handle {n_valid_handle}/{total} valid")
    if n_valid_tip < TRAJ_MIN_VALID_OBS or n_valid_handle < TRAJ_MIN_VALID_OBS:
        raise RuntimeError("Too few valid 3D observations for RTS.")

    # ── RTS Kalman gap-fill ──────────────────────────────────────────────────
    _rts_tip,    gap_tip    = _rts_fill_3d(obs_tip,    "tip")
    _rts_handle, gap_handle = _rts_fill_3d(obs_handle, "handle")
    filled_tip    = _interp_nans_3d(_rts_tip)
    filled_handle = _interp_nans_3d(_rts_handle)
    frame_ids = all_frame_ids

    # ── Phase 2: smooth + global flip (preserves tip↔handle distance) ────────
    x_tip  = smooth_1d(filled_tip[:,  0], SMOOTH_WINDOW_X)
    y_s_tip  = smooth_1d(filled_tip[:,  1], SMOOTH_WINDOW_Y)
    z_s_tip  = smooth_1d(filled_tip[:,  2], SMOOTH_WINDOW_Z) * Z_SCALE_FACTOR
    x_handle = smooth_1d(filled_handle[:, 0], SMOOTH_WINDOW_X)
    y_s_handle = smooth_1d(filled_handle[:, 1], SMOOTH_WINDOW_Y)
    z_s_handle = smooth_1d(filled_handle[:, 2], SMOOTH_WINDOW_Z) * Z_SCALE_FACTOR

    global_max_y = max(np.max(y_s_tip), np.max(y_s_handle))
    y_tip,  z_tip  = global_max_y - y_s_tip,  z_s_tip
    y_handle, z_handle = global_max_y - y_s_handle, z_s_handle

    # Save 2D static projection plots
    # save_2d_projections(x_tip, y_tip, z_tip, x_handle, y_handle, z_handle, out_dir)

    N = len(x_tip)

    # ── Phase 3: Catmull-Rom Spline Upsampling ───────────────────────────────
    if TRAJ_SPLINE_UPSAMPLE > 1:
        x_tip    = catmull_rom_spline_1d(x_tip,    num_points=TRAJ_SPLINE_UPSAMPLE)
        y_tip    = catmull_rom_spline_1d(y_tip,    num_points=TRAJ_SPLINE_UPSAMPLE)
        z_tip    = catmull_rom_spline_1d(z_tip,    num_points=TRAJ_SPLINE_UPSAMPLE)
        x_handle = catmull_rom_spline_1d(x_handle, num_points=TRAJ_SPLINE_UPSAMPLE)
        y_handle = catmull_rom_spline_1d(y_handle, num_points=TRAJ_SPLINE_UPSAMPLE)
        z_handle = catmull_rom_spline_1d(z_handle, num_points=TRAJ_SPLINE_UPSAMPLE)

    # ── Bounding box for fixed axes ──────────────────────────────────────────
    x_min = min(x_tip.min(),  x_handle.min())
    x_max = max(x_tip.max(),  x_handle.max())
    y_min = min(y_tip.min(),  y_handle.min())
    y_max = max(y_tip.max(),  y_handle.max())
    z_min = min(z_tip.min(),  z_handle.min())
    z_max = max(z_tip.max(),  z_handle.max())
    dx = max(x_max - x_min, 1.0)
    dy = max(y_max - y_min, 1.0)
    dz = max(z_max - z_min, 1.0)

    # ── Depth normalisation for dot colours ─────────────────────────────────
    # splprep/FITPACK removed: it crashed on duplicate gap-filled points.
    # The raw z_tip values produce identical colours with zero crash risk.
    z_n_tip    = (z_tip    - z_tip.min())    / (z_tip.max()    - z_tip.min()    + 1e-9)
    z_n_handle = (z_handle - z_handle.min()) / (z_handle.max() - z_handle.min() + 1e-9)
    # ── Phase 7: plane-only animated MP4s (multi-angle) ──────────────────────
    print(f"  [3d] rendering {len(TRAJ_VIEW_ANGLES)} angle(s) in parallel...")

    mp4_outs = []
    data = (x_tip, y_tip, z_tip, x_handle, y_handle, z_handle,
            z_n_tip, z_n_handle,
            x_min, x_max, y_min, y_max, z_min, z_max, dx, dy, dz)

    import concurrent.futures
    with concurrent.futures.ProcessPoolExecutor() as executor:
        for idx, view in enumerate(TRAJ_VIEW_ANGLES):
            # Support both (elev, azim) and (elev, azim, label)
            if len(view) == 3:
                elev, azim, label = view
            else:
                elev, azim = view
                label = f"elev{elev}_azim{azim}"  # fallback if no label given

            tmp_dir = tempfile.mkdtemp()
            args_list = [(i, elev, azim, data, tmp_dir) for i in range(N)]
            list(executor.map(_render_frame, args_list))

            mp4_out = os.path.join(out_dir, f"traj_3d_{label}.mp4")
            subprocess.run([
                "ffmpeg", "-y", "-loglevel", "error",
                "-framerate", str(TRAJ_FFMPEG_FRAMERATE),
                "-i", os.path.join(tmp_dir, "frame_%04d.png"),
                "-c:v", "libx264", "-pix_fmt", "yuv420p", mp4_out
            ], check=True)
            shutil.rmtree(tmp_dir)
            print(f"  [3d] saved angle {idx+1}/{len(TRAJ_VIEW_ANGLES)}: {label} -> {os.path.basename(mp4_out)}")
            mp4_outs.append(mp4_out)

    return {
        "frame_ids":  frame_ids,
        "tip_xyz":    np.column_stack([x_tip,  y_tip,  z_tip]),
        "handle_xyz": np.column_stack([x_handle, y_handle, z_handle]),
        "mp4s":       mp4_outs,
        "stats": {
            "valid_tip":         n_valid_tip,
            "valid_handle":      n_valid_handle,
            "gap_filled_tip":    gap_tip,
            "gap_filled_handle": gap_handle,
            "total_obs_frames":  total,
            "frames_3d":         N,
        },
    }



In [26]:


print("--- Stage 7: 3D Trajectory ---")
traj = build_trajectory(tip_xy, handle_xy, norm_depth, out_dir)
del tip_xy, _centroid_xy, handle_xy, norm_depth
print("Stage 7 done.")

print("--- Stage 8: Stitch Final Video ---")
try:
    mp4s = traj.get("mp4s", [])
    if len(mp4s) >= 2:
        final_out = os.path.join(out_dir, "swing_analysis.mp4")
        _make_3in1_video(rgb_path, mp4s[0], mp4s[1], final_out, label_v1="3D Front View", label_v2="3D Side View")
    elif len(mp4s) > 0:
        final_out = os.path.join(out_dir, "swing_analysis.mp4")
        _make_side_by_side(rgb_path, mp4s[0], final_out)
except Exception as e:
    print("Stitching failed:", e)

if not VISUALIZATION:
    for f in [rgb_path] + traj.get("mp4s", []):
        try:
            if os.path.exists(f): os.remove(f)
        except: pass

import subprocess
import os
from IPython.display import Video, display

# Paths
input_mp4 = os.path.join(out_dir, "swing_analysis.mp4")
play_mp4 = os.path.join(out_dir, "swing_analysis_play.mp4")

# Silently convert and display
if os.path.exists(input_mp4):
    cmd = f'ffmpeg -y -i "{input_mp4}" -vcodec libx264 -f mp4 "{play_mp4}" -loglevel quiet'
    subprocess.run(cmd, shell=True)

    if os.path.exists(play_mp4):
        rel_path = os.path.relpath(play_mp4, start=os.getcwd())
        display(Video(rel_path, embed=True, width=640))
else:
    print("❌ swing_analysis.mp4 not found. Make sure Stage 8 ran successfully.")


print(f"\n✅ All processing complete! Outputs saved in: {out_dir}")


--- Stage 7: 3D Trajectory ---
  [3d] depth fusion: tip 46/46 valid | handle 46/46 valid
  [3d] kalman: tip gap-fill 0 frames
  [3d] kalman: handle gap-fill 0 frames
  [3d] rendering 2 angle(s) in parallel...
  [3d] saved angle 1/2: front -> traj_3d_front.mp4
  [3d] saved angle 2/2: side -> traj_3d_side.mp4
Stage 7 done.
--- Stage 8: Stitch Final Video ---
  [merge] swing analysis saved: swing_analysis.mp4



✅ All processing complete! Outputs saved in: /media/user/hugeDrive2/str8-bat/adnan/Bat_plane_v2.3/notebook/bat_notebook_1/outputs
